# 段階 C6：本番（C3）の結果を強める — A：種の組を二つ足す／B：VE 型の格子を細かくする

**事前登録（`C6_事前登録.md`、確定）の後に走らせます。** 規則と走らせ方は、C3 の本番のコード（版 C3-1.0）を一字も変えずに使います。

## 何を走らせるか
- **A（二つの網）**：種の組 2 と 3 のそれぞれで、M0、WV_tele(k_best)（VP 型 k = 13、VE 型 k = 9。C3 で決まった k のまま）、U_min 2.38、U_env。5,000 枚ずつ、40 歩。
- **B（VE 型だけ）**：種の組 0 で、WV_tele の k = 10、11、12。
- 判定：種の組ごと（C3 と同じ形）と、三つの種の組（0・2・3）を合わせた判定（主、HC17〜HC24）、新しい種の組（2・3）だけの判定。B は HC25。

## 使い方
1. **Colab で開く**：ドライブに置き、右クリック →「アプリで開く」→ Google Colaboratory。
2. **GPU を選ぶ**：「ランタイム」→「ランタイムのタイプを変更」→「**L4 GPU**」。
3. **ドライブをつなぐ**：② のセルで許可します。**C3 の結果（`マイドライブ/C3_results/`）は読むだけで、書き換えません。** C6 の結果は `マイドライブ/C6_results/vp/` と `C6_results/ve/` に書きます。データと積率は `C3_results/` の直下のものを読み込み、作り直しません。
4. **走らせる順番（提案）**：① の設定を変えて、次の順に「すべてのセルを実行」します。
   1. **B**：`NET = 've'`、`RUN_A = False`、`RUN_B = True`（約 1.9 時間）
   2. **VP 型の A**：`NET = 'vp'`、`RUN_A = True`、`RUN_B = False`（約 5.8 時間）
   3. **VE 型の A**：`NET = 've'`、`RUN_A = True`、`RUN_B = False`（約 6.1 時間）
   - B を先にするのは、短くて、C5 §1.6 の読みをすぐ確かめられるからです。A は、種の組 2 を終えてから 3 に進みます。
5. **途中で切れたときの再開の仕方**：1 束（1,000 枚）ごとにドライブへ書き出しています。切れたら、**同じ設定のまま、もう一度上から実行するだけ**で、済んだ束は飛ばして続きから計算します。
6. **最後のセル（⑪）**：A と B のすべてがそろうまで、集計も図も出さずに止まります（「まだそろっていません」と出れば正しい動きです）。そろったら、集計・判定・図と `マイドライブ/C6_results/C6_report_part01.zip`（〜）を作ります。それを持ち帰ってください。

## u の曲線を見ないこと
- 種の組 2・3 の u の曲線と切り替えの σ、B の FID は、まだ誰も見ていません。このノートブックは、最後のセルの前には、それらを画面に出しません（各束の費用と時間だけを出します。切り替えの σ はログのファイルにだけ書きます）。

## 時間の見込み（C3 の L4 の実測から）
- 1 束（1,000 枚）：M0 は VP 型 632 秒・VE 型 647 秒、WV_tele は VP 型 k = 13 で 425 秒・VE 型 k = 9 で 501 秒、規則は 508〜517 秒。

| 部分 | 中身 | 時間 |
|---|---|---|
| B | VE 型 k = 10・11・12（種の組 0） | 約 1.9 時間 |
| VP 型の A | 種の組 2・3 × （M0、WV_tele(13)、U_min、U_env） | 約 5.8 時間 |
| VE 型の A | 種の組 2・3 × （M0、WV_tele(9)、U_min、U_env） | 約 6.1 時間 |
| 合わせて | | **約 14 時間** |

- このほか、セッションごとに網と積率の読み込みに数分、FID の特徴量に 1 仕事あたり 1〜2 分かかります。⑨ の最初の束の実測から、見込みを出し直して表示します。

## ライセンス
- **このノートブック、c1core.py・c2core.py・c3core.py・c6core.py**：MIT License（© 2026 Taishi Namba）。
- **EDM のコードと学習済みの重み**（NVIDIA、https://github.com/NVlabs/edm）：**CC BY-NC-SA 4.0**（非商用）。公式の場所から読み込みます。
- **AFHQv2**（NAVER、StarGAN v2）：**CC BY-NC 4.0**（非商用）。C3 で作った積率だけを使い、画像はこのノートブックでは読みません。
- **FID の Inception の重み**と**AFHQv2 64×64 の FID の基準の統計量**（EDM の公式の配布場所）：公式の場所から読み込みます。


In [ ]:
# ① 設定
NET = 've'          # 'vp' か 've'
RUN_A = False       # A：種の組 2・3（M0、WV_tele(k_best)、U_min、U_env）
RUN_B = True        # B：VE 型の WV_tele k = 10・11・12（NET = 've' のときだけ走る）
ROOT3 = '/content/drive/MyDrive/C3_results'   # C3 の結果（読むだけ）。データと積率もここ
ROOT6 = '/content/drive/MyDrive/C6_results'   # C6 の結果
NET_CHUNK = 125

EDM_BASE = 'https://nvlabs-fi-cdn.nvidia.com/edm/'
REF_URL = EDM_BASE + 'fid-refs/afhqv2-64x64.npz'
DETECTOR_URL = 'https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/metrics/inception-2015-12-05.pkl'


In [ ]:
# ② ドライブと GPU
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, pickle, gc
import numpy as np
import torch
assert torch.cuda.is_available(), 'GPU がありません。「ランタイム」→「ランタイムのタイプを変更」で L4 GPU を選んでください。'
dev = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
assert os.path.isdir(ROOT3), 'C3 の結果（C3_results）が見つかりません。'
os.makedirs(ROOT6, exist_ok=True)
print('GPU:', GPU_NAME, '  torch:', torch.__version__, '  網:', NET, '  A:', RUN_A, '  B:', RUN_B)


In [ ]:
# ③ EDM のコード（公式のリポジトリから）
!test -d /content/edm || git clone -q https://github.com/NVlabs/edm.git /content/edm
!cd /content/edm && git rev-parse HEAD > /content/edm_commit.txt
sys.path.insert(0, '/content/edm')
import dnnlib
EDM_COMMIT = open('/content/edm_commit.txt').read().strip()
print('EDM commit:', EDM_COMMIT)


In [ ]:
%%writefile /content/c1core.py
# c1core.py -- 実験場C 段階C1 の中身（MIT License, (c) 2026 Taishi Namba）
#
# 霧が晴れる時刻の合図 S_gap を、学習済みの拡散モデル（EDM の形：D(x, sigma) を返す網）で試すための道具。
# 同じコードを、Colab（torch, GPU）と作業場の CPU 確かめ（numpy）の両方で使う。
#   - 状態 x は (n, d) の配列（torch.Tensor でも numpy.ndarray でもよい）
#   - 網は net_fn(x, sigma) -> D（x と同じ型・形）。sigma は python の float。
# 他者のコード・重み（EDM, Inception）はここには入っていない。ノートブックが公式の場所から読む。

import json
import math
import os
import time

import numpy as np

C1_VERSION = "C1-1.0"

# ---------------------------------------------------------------------------
# 事前登録の定数（C1_事前登録.md の JSON と同じでなければならない。検証で照合する）
# ---------------------------------------------------------------------------
PREREG = {
    # 規則（実験場 B から変えない）
    "factor": 4.0,              # 基準値の 4 倍を超えたら切り替え
    "fog_sigma": 10.0,          # 基準値を作る霧の濃い所（sigma >= 10）
    "subset_frac": 0.05,        # 合図は束の 5% の点だけで作る
    # σ>=10 が霧の濃い所に当たらない場合の代わり（最初の 3 回の呼び出しの中央値）
    "fallback_first_calls": 3,
    # 「σ>=10 が霧の濃い所に当たるか」の判定（パイロット、M0 の軌道、50 歩）
    "pilot_steps": 50,
    "pilot_n": 256,
    "pilot_seedset": 9,
    "min_fog_calls": 3,
    "flat_band_low": [10.0, 20.0],
    "flat_band_high": [40.0, 80.0],
    "flat_ratio_max": 2.0,
    # 本番
    "main_steps": 18,
    "batch": 1000,
    "n_images_main": 10000,
    "seedset_main": 0,
    "seedset_noise": 1,
    "wv_sigma_targets": [40.0, 20.0, 10.0, 5.0, 2.5, 1.25, 0.6],
    # 判定
    "fid_rel_tol": 0.03,        # 画質を落とさない = FID <= 1.03 * FID(M0)
    "hc1_batch_frac": 0.8,      # HC1：束の 8 割以上
    "hc3_cost_max": 0.85,       # HC3：網の評価 <= 0.85 * M0
    "hc4_fid_rel": 0.03,        # HC4：FID <= 1.03 * FID(WV_tele 最良)
    "hc4_cost_margin": 0.10,    # HC4：費用 <= 費用(WV_tele 最良) + 0.10
}

# EDM の標準の目盛り
EDM_SIGMA_MIN = 0.002
EDM_SIGMA_MAX = 80.0
EDM_RHO = 7.0


# ---------------------------------------------------------------------------
# numpy / torch の小さな橋
# ---------------------------------------------------------------------------
def _is_torch(a):
    return type(a).__module__.startswith("torch")


def to_np(a):
    if _is_torch(a):
        return a.detach().cpu().numpy()
    return np.asarray(a)


def _clone(a):
    return a.clone() if _is_torch(a) else a.copy()


def _idx(idx, like):
    if _is_torch(like):
        import torch
        return torch.as_tensor(np.asarray(idx), device=like.device, dtype=torch.long)
    return np.asarray(idx)


def rownorm2(a):
    """各行の二乗ノルム（長さ n の配列）。"""
    return (a * a).reshape(a.shape[0], -1).sum(-1)


# ---------------------------------------------------------------------------
# ガウスのスコア s_G（データの平均と共分散だけ）
# ---------------------------------------------------------------------------
def moments_from_batches(batches):
    """batches: (m, d) の numpy 配列の列。float64 で平均と共分散（1/N で割る）を返す。"""
    n = 0
    s1 = None
    s2 = None
    for b in batches:
        b = np.asarray(b, dtype=np.float64)
        if s1 is None:
            s1 = np.zeros(b.shape[1])
            s2 = np.zeros((b.shape[1], b.shape[1]))
        n += b.shape[0]
        s1 += b.sum(0)
        s2 += b.T @ b
    mu = s1 / n
    cov = s2 / n - np.outer(mu, mu)
    cov = 0.5 * (cov + cov.T)
    return mu, cov, n


def eig_moments(cov):
    lam, U = np.linalg.eigh(np.asarray(cov, dtype=np.float64))
    n_neg = int((lam < 0).sum())
    lam = np.clip(lam, 0.0, None)
    return lam, U, n_neg


class GaussianModel:
    """D_G(x, s) = mu + Sigma (Sigma + s^2 I)^{-1} (x - mu)、固有値分解で計算する。

    mu, lam, U は「状態と同じ型（numpy か torch）」で持つ。to_backend で変換する。
    """

    def __init__(self, mu, lam, U, to_backend=None):
        f = to_backend if to_backend is not None else (lambda a: np.asarray(a, dtype=np.float64))
        self.mu = f(np.asarray(mu, dtype=np.float64))
        self.lam = f(np.asarray(lam, dtype=np.float64))
        self.U = f(np.asarray(U, dtype=np.float64))
        self.d = int(np.asarray(mu).shape[0])

    @classmethod
    def from_cov(cls, mu, cov, to_backend=None):
        lam, U, _ = eig_moments(cov)
        return cls(mu, lam, U, to_backend)

    def denoise(self, x, sigma):
        c = (x - self.mu) @ self.U
        c = c * (self.lam / (self.lam + float(sigma) ** 2))
        return self.mu + c @ self.U.T

    def score(self, x, sigma):
        return (self.denoise(x, sigma) - x) / float(sigma) ** 2

    def teleport(self, x, s_from, s_to):
        """ガウスの確率流 ODE を s_from から s_to まで厳密に解く（Wang & Vastola の analytical teleportation）。"""
        c = (x - self.mu) @ self.U
        c = c * ((self.lam + float(s_to) ** 2) / (self.lam + float(s_from) ** 2)) ** 0.5
        return self.mu + c @ self.U.T


# ---------------------------------------------------------------------------
# EDM の目盛りと Heun 型の決定論的サンプラー（Karras et al. 2022, Alg. 2, S_churn = 0）
# ---------------------------------------------------------------------------
def edm_sigmas(num_steps, sigma_min=EDM_SIGMA_MIN, sigma_max=EDM_SIGMA_MAX, rho=EDM_RHO):
    i = np.arange(num_steps, dtype=np.float64)
    t = (sigma_max ** (1 / rho) + i / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    return np.append(t, 0.0)


def nfe_m0(num_steps):
    return 2 * num_steps - 1


def call_sigmas(sigmas):
    """Heun の各呼び出しの (歩, sigma) の列。"""
    out = []
    N = len(sigmas) - 1
    for i in range(N):
        out.append((i, float(sigmas[i])))
        if i < N - 1:
            out.append((i, float(sigmas[i + 1])))
    return out


def heun_sample(x0, sigmas, D_call, start_step=0):
    """D_call(x, sigma, step, call_index) -> D。start_step から歩き始める（x0 は sigmas[start_step] の点）。"""
    x = x0
    N = len(sigmas) - 1
    ncall = 0
    for i in range(start_step, N):
        tc = float(sigmas[i])
        tn = float(sigmas[i + 1])
        d = (x - D_call(x, tc, i, ncall)) / tc
        ncall += 1
        xn = x + (tn - tc) * d
        if i < N - 1:
            d2 = (xn - D_call(xn, tn, i, ncall)) / tn
            ncall += 1
            xn = x + (tn - tc) * (0.5 * d + 0.5 * d2)
        x = xn
    return x


class CountingNet:
    """網の呼び出しを点の数で数える包み。"""

    def __init__(self, net_fn):
        self.net_fn = net_fn
        self.points = 0

    def __call__(self, x, sigma):
        self.points += int(x.shape[0])
        return self.net_fn(x, sigma)


# ---------------------------------------------------------------------------
# 合図 S_gap の規則（定数は PREREG）
# ---------------------------------------------------------------------------
class GapRule:
    """各呼び出しの統計量 s（束の点での中央値）を受け取り、切り替えるかを返す。

    mode = "fog"  : sigma >= fog_sigma の呼び出しの s の中央値を基準値にする。
                    sigma < fog_sigma になった最初の呼び出しで基準値を固める。以後、s > factor * 基準値 で切り替え。
    mode = "first": 最初の fallback_first_calls 回の呼び出しの s の中央値を基準値にする。
    切り替えたら後戻りしない。
    """

    def __init__(self, mode="fog", factor=None, fog_sigma=None, n_first=None):
        assert mode in ("fog", "first")
        self.mode = mode
        self.factor = PREREG["factor"] if factor is None else factor
        self.fog_sigma = PREREG["fog_sigma"] if fog_sigma is None else fog_sigma
        self.n_first = PREREG["fallback_first_calls"] if n_first is None else n_first
        self.base_vals = []
        self.baseline = None
        self.switched = False
        self.ncalls = 0

    def in_baseline_phase(self, sigma):
        if self.baseline is not None:
            return False
        if self.mode == "fog":
            return sigma >= self.fog_sigma
        return self.ncalls < self.n_first

    def observe(self, sigma, s):
        """True を返したら、この呼び出しから網に切り替える。"""
        if self.switched:
            self.ncalls += 1
            return True
        if self.in_baseline_phase(sigma):
            self.base_vals.append(float(s))
            self.ncalls += 1
            return False
        if self.baseline is None:
            if len(self.base_vals) == 0:
                raise RuntimeError("基準値を作る呼び出しが 1 つもありません（mode=%s）" % self.mode)
            self.baseline = float(np.median(self.base_vals))
        self.ncalls += 1
        if float(s) > self.factor * self.baseline:
            self.switched = True
            return True
        return False


def gap_stat_points(DG, Dn, x):
    """|s_G - s_net|^2 / |s_net|^2 = |D_G - D_net|^2 / |D_net - x|^2（sigma は消える）。点ごとの値（numpy）。"""
    num = to_np(rownorm2(DG - Dn)).astype(np.float64)
    den = to_np(rownorm2(Dn - x)).astype(np.float64)
    return num / np.maximum(den, 1e-300)


# ---------------------------------------------------------------------------
# 方法
# ---------------------------------------------------------------------------
def method_tag(m):
    if m["method"] in ("M0", "S_gap", "Gall"):
        base = m["method"]
    else:
        base = "%s_k%02d" % (m["method"], m["k"])
    return "%s_N%03d_s%d" % (base, m["steps"], m["seedset"])


def run_one(method, k, x_init, sigmas, net_fn, G, rng=None, branch="fog", rule_kw=None, sync=None):
    """一つの束を走らせる。x_init は sigma_max = sigmas[0] の点（= latents * sigmas[0]）。

    返り値：(x_final, info)。info["cost_units"] は「束全体で網 1 回」を単位にした網の評価の回数。
    """
    n = int(x_init.shape[0])
    net = CountingNet(net_fn)
    N = len(sigmas) - 1
    info = {"method": method, "k": k, "n": n, "steps": N, "branch": None}
    if sync:
        sync()
    t0 = time.time()

    if method == "M0":
        x = heun_sample(x_init, sigmas, lambda x, s, i, c: net(x, s))

    elif method == "Gall":
        x = heun_sample(x_init, sigmas, lambda x, s, i, c: G.denoise(x, s))

    elif method == "WV_step":
        # 最初の k 歩（その歩の補正の呼び出しも含む）を D_G で進み、あとは網
        x = heun_sample(x_init, sigmas, lambda x, s, i, c: G.denoise(x, s) if i < k else net(x, s))
        info["switch_sigma"] = float(sigmas[k])

    elif method == "WV_tele":
        # sigma_max から sigmas[k] までガウスで厳密に飛ばし、そこから網で Heun
        xk = G.teleport(x_init, float(sigmas[0]), float(sigmas[k])) if k > 0 else x_init
        x = heun_sample(xk, sigmas, lambda x, s, i, c: net(x, s), start_step=k)
        info["switch_sigma"] = float(sigmas[k])

    elif method == "S_gap":
        if rng is None:
            rng = np.random.default_rng(0)
        kw = dict(rule_kw or {})
        frac = kw.pop("subset_frac", PREREG["subset_frac"])
        rule = GapRule(mode=branch, **kw)
        m = max(1, int(math.ceil(frac * n)))
        trace = []
        state = {"switch_sigma": None, "switch_call": None}

        def D_call(x, s, i, c):
            if rule.switched:
                rule.observe(s, float("nan"))
                return net(x, s)
            DG = G.denoise(x, s)
            sub = np.sort(rng.choice(n, size=m, replace=False))
            isub = _idx(sub, x)
            xs = x[isub]
            Dn_sub = net(xs, s)
            st = float(np.median(gap_stat_points(DG[isub], Dn_sub, xs)))
            dd = float(np.median(to_np(rownorm2(DG[isub] - Dn_sub)).astype(np.float64))) / G.d
            switch = rule.observe(s, st)
            trace.append([c, float(s), st, dd])  # 呼び出し, σ, 合図の統計量, 中央値 |D_G − D_net|²/d（観察用）
            out = _clone(DG)
            out[isub] = Dn_sub
            if switch:
                rest = np.setdiff1d(np.arange(n), sub)
                if len(rest) > 0:
                    irest = _idx(rest, x)
                    out[irest] = net(x[irest], s)
                state["switch_sigma"] = float(s)
                state["switch_call"] = int(c)
            return out

        x = heun_sample(x_init, sigmas, D_call)
        info.update({
            "branch": branch, "subset_m": m, "trace": trace, "baseline": rule.baseline,
            "switched": rule.switched,
            "switch_sigma": state["switch_sigma"] if rule.switched else 0.0,
            "switch_call": state["switch_call"],
        })
    else:
        raise ValueError(method)

    if sync:
        sync()
    info["seconds"] = time.time() - t0
    info["net_points"] = net.points
    info["cost_units"] = net.points / n
    info["cost_ratio"] = info["cost_units"] / nfe_m0(N)
    return x, info


# ---------------------------------------------------------------------------
# パイロット：M0 の軌道に沿って、各呼び出しで全点の統計量と |D_G - D_net|^2 を測る
# ---------------------------------------------------------------------------
def pilot_curve(x_init, sigmas, net_fn, G):
    rows = []

    def D_call(x, s, i, c):
        Dn = net_fn(x, s)
        DG = G.denoise(x, s)
        st = gap_stat_points(DG, Dn, x)
        dd = to_np(rownorm2(DG - Dn)).astype(np.float64) / G.d
        rows.append({"call": int(c), "step": int(i), "sigma": float(s),
                     "stat_median": float(np.median(st)),
                     "stat_q25": float(np.quantile(st, 0.25)), "stat_q75": float(np.quantile(st, 0.75)),
                     "dD2_per_dim_median": float(np.median(dd)), "dD2_per_dim_mean": float(np.mean(dd))})
        return Dn

    heun_sample(x_init, sigmas, D_call)
    return rows


def decide_branch(rows, P=PREREG):
    """事前登録の規則で、σ>=10 が霧の濃い所に当たるかを決める。"""
    s = np.array([r["sigma"] for r in rows])
    v = np.array([r["stat_median"] for r in rows])
    fog = s >= P["fog_sigma"]
    res = {"n_fog_calls": int(fog.sum()), "sigma_max": float(s.max())}
    lo = (s >= P["flat_band_low"][0]) & (s <= P["flat_band_low"][1])
    hi = (s >= P["flat_band_high"][0]) & (s <= P["flat_band_high"][1])
    res["n_low"] = int(lo.sum())
    res["n_high"] = int(hi.sum())
    ok_range = res["sigma_max"] > P["fog_sigma"] and res["n_fog_calls"] >= P["min_fog_calls"]
    if lo.sum() == 0 or hi.sum() == 0 or not ok_range:
        res.update({"ratio_low_over_high": None, "flat": False, "branch": "first",
                    "reason": "sigma の範囲が足りない"})
        return res
    ratio = float(np.median(v[lo]) / np.median(v[hi]))
    flat = (1.0 / P["flat_ratio_max"]) <= ratio <= P["flat_ratio_max"]
    res.update({"ratio_low_over_high": ratio, "flat": bool(flat),
                "branch": "fog" if flat else "first",
                "reason": "平ら" if flat else "平らでない（基準値を最初の %d 回で作る）" % P["fallback_first_calls"]})
    return res


# ---------------------------------------------------------------------------
# 書き出し（1 束 1 ファイル、途中で切れても再開できる）
# ---------------------------------------------------------------------------
def _atomic_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)


def _atomic_npz(path, **arrs):
    tmp = path + ".tmp.npz"
    np.savez_compressed(tmp, **arrs)
    os.replace(tmp, path)


def to_uint8_images(x, img_shape):
    a = to_np(x).astype(np.float64).reshape((-1,) + tuple(img_shape))
    return np.clip(a * 127.5 + 128.0, 0, 255).astype(np.uint8)


def wv_k_grid(sigmas, targets=None):
    targets = PREREG["wv_sigma_targets"] if targets is None else targets
    t = np.asarray(sigmas[1:-1])  # k = 1..N-1
    ks = set()
    for tg in targets:
        ks.add(int(np.argmin(np.abs(np.log(t) - math.log(tg)))) + 1)
    return sorted(ks)


def main_jobs(steps=None, seedset=None, ks=None, with_noise=True):
    steps = PREREG["main_steps"] if steps is None else steps
    seedset = PREREG["seedset_main"] if seedset is None else seedset
    ks = wv_k_grid(edm_sigmas(steps)) if ks is None else ks
    jobs = [{"method": "M0", "k": 0, "steps": steps, "seedset": seedset},
            {"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}]
    for k in ks:
        jobs.append({"method": "WV_tele", "k": k, "steps": steps, "seedset": seedset})
    for k in ks:
        jobs.append({"method": "WV_step", "k": k, "steps": steps, "seedset": seedset})
    if with_noise:
        jobs.append({"method": "M0", "k": 0, "steps": steps, "seedset": PREREG["seedset_noise"]})
    return jobs


def subset_rng(seedset, steps, b):
    return np.random.default_rng([20261112, int(seedset), int(steps), int(b)])


def phase_pilot(out, net_fn, G, make_latents, n=None, steps=None, seedset=None, log=print):
    n = PREREG["pilot_n"] if n is None else n
    steps = PREREG["pilot_steps"] if steps is None else steps
    seedset = PREREG["pilot_seedset"] if seedset is None else seedset
    os.makedirs(out, exist_ok=True)
    p = os.path.join(out, "pilot.json")
    if os.path.exists(p):
        log("パイロットは済んでいます：" + p)
        with open(p, encoding="utf-8") as f:
            return json.load(f)["decision"]
    sig = edm_sigmas(steps)
    x0 = make_latents(seedset, 0, n) * float(sig[0])
    rows = pilot_curve(x0, sig, net_fn, G)
    dec = decide_branch(rows)
    _atomic_json(p, {"version": C1_VERSION, "steps": steps, "n": n, "seedset": seedset,
                     "rows": rows, "decision": dec, "prereg": PREREG})
    log("パイロットの判定：%s（%s）" % (dec["branch"], dec["reason"]))
    return dec


def load_branch(out):
    with open(os.path.join(out, "pilot.json"), encoding="utf-8") as f:
        return json.load(f)["decision"]["branch"]


def phase_runs(out, jobs, net_fn, G, make_latents, n_images=None, batch=None, img_shape=(3, 32, 32),
               branch=None, sync=None, log=print):
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    if branch is None:
        branch = load_branch(out)
    n_batches = int(math.ceil(n_images / batch))
    for job in jobs:
        tag = method_tag(job)
        d = os.path.join(out, "runs", tag)
        os.makedirs(d, exist_ok=True)
        sig = edm_sigmas(job["steps"])
        for b in range(n_batches):
            fi = os.path.join(d, "b%03d.npz" % b)
            fj = os.path.join(d, "b%03d.json" % b)
            if os.path.exists(fi) and os.path.exists(fj):
                continue
            nb = min(batch, n_images - b * batch)
            x0 = make_latents(job["seedset"], b, nb) * float(sig[0])
            x, info = run_one(job["method"], job["k"], x0, sig, net_fn, G,
                              rng=subset_rng(job["seedset"], job["steps"], b), branch=branch, sync=sync)
            info.update({"tag": tag, "batch": b, "seedset": job["seedset"], "version": C1_VERSION})
            _atomic_npz(fi, images=to_uint8_images(x, img_shape))
            _atomic_json(fj, info)
            msg = "%s b%03d  費用 %.3f × M0  %.1f 秒" % (tag, b, info["cost_ratio"], info["seconds"])
            if job["method"] == "S_gap":
                msg += "  切り替え σ = %.4g" % info["switch_sigma"]
            log(msg)
            with open(os.path.join(out, "log.txt"), "a", encoding="utf-8") as f:
                f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + msg + "\n")


# ---------------------------------------------------------------------------
# FID
# ---------------------------------------------------------------------------
def fid_from_stats(mu, sigma, mu_ref, sigma_ref):
    """EDM の fid.py と同じ式。"""
    import scipy.linalg
    m = np.square(mu - mu_ref).sum()
    try:
        s, _ = scipy.linalg.sqrtm(np.dot(sigma, sigma_ref), disp=False)
    except TypeError:  # 新しい scipy には disp がない
        s = scipy.linalg.sqrtm(np.dot(sigma, sigma_ref))
    return float(np.real(m + np.trace(sigma + sigma_ref - s * 2)))


def feat_stats(feats):
    feats = np.asarray(feats, dtype=np.float64)
    return feats.mean(0), np.cov(feats, rowvar=False)


def run_complete(out, tag, n_images=None, batch=None):
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    d = os.path.join(out, "runs", tag)
    nb = int(math.ceil(n_images / batch))
    return all(os.path.exists(os.path.join(d, "b%03d.npz" % b)) for b in range(nb))


def phase_features(out, tags, feat_fn, n_images=None, batch=None, log=print):
    """feat_fn(uint8 の NCHW numpy) -> (m, F) numpy。方法ごとに feats.npy を書く。"""
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    nb = int(math.ceil(n_images / batch))
    for tag in tags:
        d = os.path.join(out, "runs", tag)
        fp = os.path.join(d, "feats_%d.npy" % n_images)
        if os.path.exists(fp) or not run_complete(out, tag, n_images, batch):
            continue
        fs = []
        for b in range(nb):
            with np.load(os.path.join(d, "b%03d.npz" % b)) as z:
                fs.append(np.asarray(feat_fn(z["images"]), dtype=np.float64))
        np.save(fp + ".tmp.npy", np.concatenate(fs, 0))
        os.replace(fp + ".tmp.npy", fp)
        log("特徴量：%s" % tag)


# ---------------------------------------------------------------------------
# 集計と判定
# ---------------------------------------------------------------------------
def collect(out, n_images=None, batch=None, ref=None):
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    nb = int(math.ceil(n_images / batch))
    res = {}
    rd = os.path.join(out, "runs")
    if not os.path.isdir(rd):
        return res
    for tag in sorted(os.listdir(rd)):
        d = os.path.join(rd, tag)
        infos = []
        for b in range(nb):
            fj = os.path.join(d, "b%03d.json" % b)
            if os.path.exists(fj):
                with open(fj, encoding="utf-8") as f:
                    infos.append(json.load(f))
        if not infos:
            continue
        r = {"method": infos[0]["method"], "k": infos[0]["k"], "steps": infos[0]["steps"],
             "seedset": infos[0]["seedset"], "n_batches_done": len(infos), "complete": len(infos) == nb,
             "cost_ratio_mean": float(np.mean([i["cost_ratio"] for i in infos])),
             "seconds_total": float(np.sum([i["seconds"] for i in infos]))}
        if "switch_sigma" in infos[0]:
            sw = [float(i["switch_sigma"]) for i in infos]
            r["switch_sigmas"] = sw
            r["switch_sigma_median"] = float(np.median(sw))
        if r["method"] == "S_gap":
            r["baselines"] = [i["baseline"] for i in infos]
            r["branch"] = infos[0]["branch"]
        fp = os.path.join(d, "feats_%d.npy" % n_images)
        if ref is not None and os.path.exists(fp):
            mu, sg = feat_stats(np.load(fp))
            r["fid"] = fid_from_stats(mu, sg, ref[0], ref[1])
        res[tag] = r
    return res


def judge(res, steps=None, seedset=None, P=PREREG):
    """HC1〜HC4 を事前登録の基準で判定する。足りないものがあれば「未判定」。"""
    steps = P["main_steps"] if steps is None else steps
    seedset = P["seedset_main"] if seedset is None else seedset
    sig = edm_sigmas(steps)
    T = lambda m, k=0: method_tag({"method": m, "k": k, "steps": steps, "seedset": seedset})
    out = {"steps": steps, "seedset": seedset}
    m0 = res.get(T("M0"))
    sg = res.get(T("S_gap"))
    if not m0 or "fid" not in m0 or not sg or "fid" not in sg:
        out["status"] = "未判定（M0 か S_gap の FID がまだありません）"
        return out
    tol = (1 + P["fid_rel_tol"]) * m0["fid"]
    out["fid_M0"] = m0["fid"]
    out["fid_tol"] = tol
    out["fid_S_gap"] = sg["fid"]
    out["cost_S_gap"] = sg["cost_ratio_mean"]
    out["switch_sigmas_S_gap"] = sg["switch_sigmas"]

    def grid(meth):
        g = []
        for tag, r in res.items():
            if r["method"] == meth and r["steps"] == steps and r["seedset"] == seedset and "fid" in r:
                g.append((r["k"], r["fid"], r["cost_ratio_mean"]))
        return sorted(g)

    gs = grid("WV_step")
    gt = grid("WV_tele")
    out["grid_WV_step"] = gs
    out["grid_WV_tele"] = gt

    # HC1：WV_step の格子で、小さい k から続けて「画質を落とさない」最大の k
    kstar = 0
    for k, f, c in gs:
        if f <= tol:
            kstar = k
        else:
            break
    out["k_star_WV_step"] = kstar
    if not gs:
        out["HC1"] = "未判定（WV_step の格子がありません）"
    elif kstar == 0:
        out["sigma_safe"] = None
        out["HC1"] = "不支持（WV_step で画質を落とさず飛ばせる k がない）"
    else:
        ss = float(sig[kstar])
        out["sigma_safe"] = ss
        sw = np.asarray(sg["switch_sigmas"])
        frac = float(np.mean((sw >= ss * (1 - 1e-9)) & (sw > 0)))
        out["HC1_frac"] = frac
        out["HC1"] = "支持" if frac >= P["hc1_batch_frac"] else "不支持"
    # HC2
    out["HC2"] = "支持" if sg["fid"] <= tol else "不支持"
    # HC3
    out["HC3"] = "支持" if sg["cost_ratio_mean"] <= P["hc3_cost_max"] else "不支持"
    # HC4：WV_tele の格子で画質を落とさない k のうち最も安い（最大の k）
    safe_t = [(k, f, c) for (k, f, c) in gt if f <= tol]
    if not gt:
        out["HC4"] = "未判定（WV_tele の格子がありません）"
    else:
        if safe_t:
            kb, fb, cb = max(safe_t, key=lambda z: z[0])
        else:
            kb, fb, cb = 0, m0["fid"], 1.0
        out["WV_tele_best"] = {"k": kb, "fid": fb, "cost": cb, "sigma": float(sig[kb])}
        ok = sg["fid"] <= (1 + P["hc4_fid_rel"]) * fb and sg["cost_ratio_mean"] <= cb + P["hc4_cost_margin"]
        out["HC4"] = "支持" if ok else "不支持"
    noise = res.get(method_tag({"method": "M0", "k": 0, "steps": steps, "seedset": P["seedset_noise"]}))
    if noise and "fid" in noise:
        out["fid_M0_other_seeds"] = noise["fid"]
    out["status"] = "判定済み"
    return out


# ---------------------------------------------------------------------------
# 図
# ---------------------------------------------------------------------------
def make_figures(out, res, judged, steps=None, seedset=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    steps = PREREG["main_steps"] if steps is None else steps
    seedset = PREREG["seedset_main"] if seedset is None else seedset
    fd = os.path.join(out, "figures")
    os.makedirs(fd, exist_ok=True)
    files = []
    # 図 1：パイロットの |D_G - D_net|^2 と合図の統計量、切り替えの位置
    pp = os.path.join(out, "pilot.json")
    if os.path.exists(pp):
        with open(pp, encoding="utf-8") as f:
            pil = json.load(f)
        rows = pil["rows"]
        s = np.array([r["sigma"] for r in rows])
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].loglog(s, [r["dD2_per_dim_median"] for r in rows], "o-", ms=3, label="median |D_G - D_net|^2 / d")
        ax[1].loglog(s, [r["stat_median"] for r in rows], "o-", ms=3, label="S_gap stat (median)")
        sgr = res.get(method_tag({"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}))
        for a in ax:
            a.axvline(PREREG["fog_sigma"], color="gray", ls=":", label="sigma = 10")
            if sgr:
                for i, sw in enumerate(sgr["switch_sigmas"]):
                    if sw > 0:
                        a.axvline(sw, color="C3", alpha=0.3, label="S_gap switch" if i == 0 else None)
            if judged.get("sigma_safe"):
                a.axvline(judged["sigma_safe"], color="C2", ls="--", label="sigma_safe (WV_step)")
            a.invert_xaxis()
            a.set_xlabel("sigma")
            a.legend(fontsize=7)
        ax[1].set_title("branch: %s" % pil["decision"]["branch"])
        ax[0].set_title("pilot (M0 trajectories, %d steps)" % pil["steps"])
        fig.tight_layout()
        p = os.path.join(fd, "fig1_gaussian_gap_and_switch.png")
        fig.savefig(p, dpi=130)
        plt.close(fig)
        files.append(p)
    # 図 2：FID と費用
    fig, ax = plt.subplots(figsize=(6, 4.5))
    any_pt = False
    for meth, mk in (("WV_tele", "s-"), ("WV_step", "^-")):
        g = sorted([(r["cost_ratio_mean"], r["fid"]) for r in res.values()
                    if r["method"] == meth and r["steps"] == steps and r["seedset"] == seedset and "fid" in r])
        if g:
            ax.plot([a for a, b in g], [b for a, b in g], mk, label=meth)
            any_pt = True
    for meth, mk in (("M0", "ko"), ("S_gap", "r*")):
        r = res.get(method_tag({"method": meth, "k": 0, "steps": steps, "seedset": seedset}))
        if r and "fid" in r:
            ax.plot([r["cost_ratio_mean"]], [r["fid"]], mk, ms=11, label=meth)
            any_pt = True
    if judged.get("fid_tol"):
        ax.axhline(judged["fid_tol"], color="gray", ls=":", label="1.03 x FID(M0)")
    if any_pt:
        ax.set_xlabel("net evaluations / M0")
        ax.set_ylabel("FID (%d images)" % PREREG["n_images_main"])
        ax.legend(fontsize=8)
        fig.tight_layout()
        p = os.path.join(fd, "fig2_fid_vs_cost.png")
        fig.savefig(p, dpi=130)
        files.append(p)
    plt.close(fig)
    # 図 3：S_gap の統計量の軌跡（束ごと）
    d = os.path.join(out, "runs", method_tag({"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}))
    if os.path.isdir(d):
        fig, ax = plt.subplots(figsize=(6, 4.5))
        for fj in sorted(x for x in os.listdir(d) if x.endswith(".json")):
            with open(os.path.join(d, fj), encoding="utf-8") as f:
                inf = json.load(f)
            tr = np.array(inf["trace"]) if inf["trace"] else np.zeros((0, 3))
            if len(tr):
                ax.loglog(tr[:, 1], tr[:, 2], ".-", alpha=0.5)
            if inf["baseline"]:
                ax.axhline(PREREG["factor"] * inf["baseline"], color="C3", alpha=0.2)
        ax.invert_xaxis()
        ax.set_xlabel("sigma")
        ax.set_ylabel("stat (median over 5% subset)")
        ax.set_title("S_gap traces; red = 4 x baseline")
        fig.tight_layout()
        p = os.path.join(fd, "fig3_sgap_traces.png")
        fig.savefig(p, dpi=130)
        plt.close(fig)
        files.append(p)
    return files


def phase_report(out, ref, steps=None, seedset=None, n_images=None, batch=None, extra=None, log=print):
    res = collect(out, n_images=n_images, batch=batch, ref=ref)
    judged = judge(res, steps=steps, seedset=seedset)
    figs = make_figures(out, res, judged, steps=steps, seedset=seedset)
    rep = os.path.join(out, "report")
    os.makedirs(rep, exist_ok=True)
    summ = {"version": C1_VERSION, "prereg": PREREG, "results": res, "judgement": judged,
            "figures": [os.path.basename(f) for f in figs]}
    if extra:
        summ["extra"] = extra
    _atomic_json(os.path.join(rep, "summary_C1.json"), summ)
    for k in ("HC1", "HC2", "HC3", "HC4"):
        if k in judged:
            log("%s：%s" % (k, judged[k]))
    log("状態：%s" % judged.get("status"))
    return summ


# ---------------------------------------------------------------------------
# 持ち帰る結果を zip にする（画像の npz と特徴量は入れない。1 つ 100 ファイル以下、パスを保つ）
# ---------------------------------------------------------------------------
def pack_report(out, name="C1_report", max_files=100, top="C1_results"):
    import zipfile
    files = []
    for dp, dn, fn in os.walk(out):
        dn.sort()
        for f in sorted(fn):
            if f.endswith((".json", ".png", ".txt")) and ".tmp" not in f:
                files.append(os.path.join(dp, f))
    for old in os.listdir(out):
        if old.startswith(name + "_part") and old.endswith(".zip"):
            os.remove(os.path.join(out, old))
    zips = []
    for i in range(0, len(files), max_files):
        zp = os.path.join(out, "%s_part%02d.zip" % (name, i // max_files + 1))
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            for f in files[i:i + max_files]:
                z.write(f, os.path.join(top, os.path.relpath(f, out)))
        zips.append(zp)
    return zips, len(files)


In [ ]:
%%writefile /content/c2core.py
# c2core.py -- 実験場C 段階C2 の中身（MIT License, (c) 2026 Taishi Namba）
#
# 正規化しない合図 U_min を、学習済みの EDM（VP と VE の網）で試す。
#   u = 束の 5% の点での |D_G − D_net|² / d の中央値（呼び出しごと、時間の間引きなし = T1P5）
#   U_min（B21_rule_for_C.md §3 の定義どおり）：
#          k ≥ 1 で u_k ≥ C_MIN × min(u_0, …, u_{k−1}) となった最初の評価で、全部の点を網に切り替える
#          （1 回目の評価では切り替えない。後戻りしない。切り替える前は、全部の点を D_G で進める）
# c1core.py（C1 で Colab が走らせたものと一字一句同じ）を import し、手を入れない。

import json
import math
import os
import time

import numpy as np

import c1core as C1

C2_VERSION = "C2-1.1"

EDM_BASE = "https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/"

# ---------------------------------------------------------------------------
# 事前登録の定数（C2_事前登録.md の JSON と同じでなければならない。検証で照合する）
# ---------------------------------------------------------------------------
PREREG2 = {
    "c_min_main": 2.38,
    "c_min_sub": 7.35,
    "subset_frac": 0.05,
    "time_thin": 1,
    "steps": 18,
    "batch": 1000,
    "n_images": 10000,
    "seedset_main": 0,
    "seedset_noise": 1,
    "wv_sigma_targets": [40.0, 20.0, 10.0, 5.0, 2.5, 1.25, 0.6],
    "fid_rel_tol": 0.03,
    "cost_max": 0.85,
    "grid_fid_rel": 0.03,
    "grid_cost_margin": 0.10,
    "vp_model": "edm-cifar10-32x32-uncond-vp.pkl",
    "ve_model": "edm-cifar10-32x32-uncond-ve.pkl",
    "sampler_sigma_min": 0.002,
    "sampler_sigma_max": 80.0,
    "sampler_rho": 7.0,
    "replication_rel_tol": 1e-4,
    "rule_inequality": ">=",
    "pre_switch_update": "D_G for all points (B21)",
    "subset_count": "max(1, round(0.05 * n))",
}


# ---------------------------------------------------------------------------
# サンプラーの目盛り（EDM の edm_sampler と同じ決め方）
# ---------------------------------------------------------------------------
def sampler_range(net_sigma_min, net_sigma_max, P=PREREG2):
    """edm_sampler：sigma_min = max(0.002, net.sigma_min)、sigma_max = min(80, net.sigma_max)。"""
    return max(P["sampler_sigma_min"], float(net_sigma_min)), min(P["sampler_sigma_max"], float(net_sigma_max))


def sigmas_for(steps, srange=None, P=PREREG2):
    lo, hi = srange if srange is not None else (P["sampler_sigma_min"], P["sampler_sigma_max"])
    return C1.edm_sigmas(steps, sigma_min=lo, sigma_max=hi, rho=P["sampler_rho"])


# ---------------------------------------------------------------------------
# 規則 U_min
# ---------------------------------------------------------------------------
class UMinRule:
    """observe(u) が True を返したら、その評価から網に切り替える。

    B21_rule_for_C.md §3 の定義どおり（評価の番号 k = 0, 1, 2, …）。
    - k = 0（1 回目の評価）では切り替えない（最小値がまだない）。
    - k ≥ 1：u_k ≥ c_min × min(u_0, …, u_{k−1}) なら切り替え（等しい場合も切り替える）。
      （c_min ≥ 1 では、最小値に今の評価を含めても含めなくても同じ判定になる。）
    - 切り替えたら後戻りしない。
    """

    def __init__(self, c_min):
        self.c_min = float(c_min)
        self.min = None
        self.n_eval = 0
        self.switched = False
        self.min_at_switch = None

    def observe(self, u):
        if self.switched:
            return True
        u = float(u)
        self.n_eval += 1
        if self.min is not None and u >= self.c_min * self.min:
            self.switched = True
            self.min_at_switch = self.min
            return True
        self.min = u if self.min is None else min(self.min, u)
        return False


def replay_umin(us, c_min):
    """u の列に規則を当て、切り替えた評価の番号（0 始まり）を返す。切り替えなければ None。"""
    r = UMinRule(c_min)
    for i, u in enumerate(us):
        if r.observe(u):
            return i
    return None


def u_points(DG, Dn):
    """点ごとの |D_G − D_net|² / d（numpy）。"""
    d = int(np.prod(DG.shape[1:]))
    return C1.to_np(C1.rownorm2(DG - Dn)).astype(np.float64) / d


# ---------------------------------------------------------------------------
# 一つの束を U_min で走らせる（T1P5）
# ---------------------------------------------------------------------------
def run_umin(c_min, x_init, sigmas, net_fn, G, rng, frac=None, sync=None):
    frac = PREREG2["subset_frac"] if frac is None else frac
    n = int(x_init.shape[0])
    net = C1.CountingNet(net_fn)
    N = len(sigmas) - 1
    rule = UMinRule(c_min)
    m = max(1, int(round(frac * n)))  # B21：round(0.05 × 束の大きさ)、少なくとも 1 点
    trace = []
    state = {"switch_sigma": None, "switch_call": None}
    if sync:
        sync()
    t0 = time.time()

    def D_call(x, s, i, c):
        if rule.switched:
            return net(x, s)
        DG = G.denoise(x, s)
        sub = np.sort(rng.choice(n, size=m, replace=False))
        isub = C1._idx(sub, x)
        xs = x[isub]
        Dn_sub = net(xs, s)
        DGs = DG[isub]
        u = float(np.median(u_points(DGs, Dn_sub)))
        gap = float(np.median(C1.gap_stat_points(DGs, Dn_sub, xs)))
        prev_min = rule.min
        switch = rule.observe(u)
        trace.append([int(c), float(s), u, gap, prev_min])
        if not switch:
            # B21：切り替える前は、全部の点を D_G で進める（網を評価した 5% の点も）
            return DG
        out = C1._clone(DG)
        out[isub] = Dn_sub
        rest = np.setdiff1d(np.arange(n), sub)
        if len(rest) > 0:
            irest = C1._idx(rest, x)
            out[irest] = net(x[irest], s)
        state["switch_sigma"] = float(s)
        state["switch_call"] = int(c)
        return out

    x = C1.heun_sample(x_init, sigmas, D_call)
    if sync:
        sync()
    info = {"method": "U_min", "k": 0, "c_min": float(c_min), "n": n, "steps": N, "subset_m": m,
            "trace": trace, "trace_columns": ["call", "sigma", "u", "gap_stat", "min_before"],
            "switched": rule.switched,
            "switch_sigma": state["switch_sigma"] if rule.switched else 0.0,
            "switch_call": state["switch_call"], "min_at_switch": rule.min_at_switch,
            "seconds": time.time() - t0, "net_points": net.points}
    info["cost_units"] = net.points / n
    info["cost_ratio"] = info["cost_units"] / C1.nfe_m0(N)
    return x, info


# ---------------------------------------------------------------------------
# 仕事の名前と一覧
# ---------------------------------------------------------------------------
def tag(job):
    if job["method"] == "U_min":
        return "U_min_c%s_N%03d_s%d" % (("%g" % job["c_min"]), job["steps"], job["seedset"])
    return C1.method_tag(job)


def umin_jobs(cs=None, steps=None, seedset=None):
    cs = [PREREG2["c_min_main"], PREREG2["c_min_sub"]] if cs is None else cs
    steps = PREREG2["steps"] if steps is None else steps
    seedset = PREREG2["seedset_main"] if seedset is None else seedset
    return [{"method": "U_min", "k": 0, "c_min": c, "steps": steps, "seedset": seedset} for c in cs]


def ve_base_jobs(sigmas, steps=None):
    """規則に関わらない仕事（M0、WV_tele の格子、別の種の M0）。安い k から。"""
    steps = PREREG2["steps"] if steps is None else steps
    s0 = PREREG2["seedset_main"]
    ks = C1.wv_k_grid(sigmas, PREREG2["wv_sigma_targets"])
    jobs = [{"method": "M0", "k": 0, "steps": steps, "seedset": s0}]
    jobs += [{"method": "WV_tele", "k": k, "steps": steps, "seedset": s0} for k in sorted(ks, reverse=True)]
    jobs += [{"method": "M0", "k": 0, "steps": steps, "seedset": PREREG2["seedset_noise"]}]
    return jobs


def ve_rule_jobs(steps=None):
    """規則の仕事：主の U_min（2.38）、副の U_min（7.35）、S_gap（C1 の形）。"""
    steps = PREREG2["steps"] if steps is None else steps
    s0 = PREREG2["seedset_main"]
    jobs = umin_jobs(steps=steps)
    jobs.append({"method": "S_gap", "k": 0, "steps": steps, "seedset": s0})
    return jobs


# ---------------------------------------------------------------------------
# 走らせる（1 束 1 ファイル、再開できる）
# ---------------------------------------------------------------------------
def phase_runs2(out, jobs, net_fn, G, make_latents, srange=None, branch=None, n_images=None, batch=None,
                img_shape=(3, 32, 32), sync=None, log=print):
    n_images = PREREG2["n_images"] if n_images is None else n_images
    batch = PREREG2["batch"] if batch is None else batch
    nb = int(math.ceil(n_images / batch))
    for job in jobs:
        tg = tag(job)
        d = os.path.join(out, "runs", tg)
        os.makedirs(d, exist_ok=True)
        sig = sigmas_for(job["steps"], srange)
        for b in range(nb):
            fi = os.path.join(d, "b%03d.npz" % b)
            fj = os.path.join(d, "b%03d.json" % b)
            if os.path.exists(fi) and os.path.exists(fj):
                continue
            n = min(batch, n_images - b * batch)
            x0 = make_latents(job["seedset"], b, n) * float(sig[0])
            rng = C1.subset_rng(job["seedset"], job["steps"], b)
            if job["method"] == "U_min":
                x, info = run_umin(job["c_min"], x0, sig, net_fn, G, rng, sync=sync)
            else:
                if job["method"] == "S_gap" and branch is None:
                    raise RuntimeError("S_gap には枝（パイロットの判定）が要ります")
                x, info = C1.run_one(job["method"], job["k"], x0, sig, net_fn, G, rng=rng,
                                     branch=branch or "fog", sync=sync)
            info.update({"tag": tg, "batch": b, "seedset": job["seedset"], "version": C2_VERSION,
                         "sigma_range": [float(sig[-2]), float(sig[0])]})
            C1._atomic_npz(fi, images=C1.to_uint8_images(x, img_shape))
            C1._atomic_json(fj, info)
            msg = "%s b%03d  費用 %.3f × M0  %.1f 秒" % (tg, b, info["cost_ratio"], info["seconds"])
            if "switch_sigma" in info and job["method"] in ("U_min", "S_gap"):
                msg += "  切り替え σ = %.4g" % info["switch_sigma"]
            log(msg)
            with open(os.path.join(out, "log.txt"), "a", encoding="utf-8") as f:
                f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + msg + "\n")


def phase_pilot2(out, net_fn, G, make_latents, srange=None, log=print):
    """C1 と同じパイロット（50 歩、256 枚、種の組 9）。S_gap の枝を C1 の規則で決める。"""
    P = C1.PREREG
    os.makedirs(out, exist_ok=True)
    p = os.path.join(out, "pilot.json")
    if os.path.exists(p):
        with open(p, encoding="utf-8") as f:
            return json.load(f)["decision"]
    sig = sigmas_for(P["pilot_steps"], srange)
    x0 = make_latents(P["pilot_seedset"], 0, P["pilot_n"]) * float(sig[0])
    rows = C1.pilot_curve(x0, sig, net_fn, G)
    dec = C1.decide_branch(rows)
    C1._atomic_json(p, {"version": C2_VERSION, "steps": P["pilot_steps"], "n": P["pilot_n"],
                        "seedset": P["pilot_seedset"], "rows": rows, "decision": dec,
                        "sigma_range": [float(sig[-2]), float(sig[0])]})
    log("パイロット（S_gap の枝）：%s（%s）" % (dec["branch"], dec["reason"]))
    return dec


# ---------------------------------------------------------------------------
# 集計
# ---------------------------------------------------------------------------
def collect2(out, ref=None, n_images=None, batch=None):
    n_images = PREREG2["n_images"] if n_images is None else n_images
    batch = PREREG2["batch"] if batch is None else batch
    res = C1.collect(out, n_images=n_images, batch=batch, ref=ref)
    for tg, r in res.items():
        if r["method"] == "U_min":
            with open(os.path.join(out, "runs", tg, "b000.json"), encoding="utf-8") as f:
                r["c_min"] = json.load(f)["c_min"]
            r["switched_all"] = r["switch_sigmas"] and all(s > 0 for s in r["switch_sigmas"])
    return res


def judge_rule(res, rule_tag, steps=None, seedset=None, P=PREREG2, srange=None):
    """規則（U_min など）の二つの判定。
    a：FID ≤ 1.03 × FID(M0)、かつ 費用 ≤ 0.85 × M0
    b：WV_tele の格子で「画質を落とさない」最も安い k（なければ M0）と比べ、
       FID ≤ 1.03 × FID(WV_tele(k_best))、かつ 費用 ≤ 費用(k_best) + 0.10
    """
    steps = P["steps"] if steps is None else steps
    seedset = P["seedset_main"] if seedset is None else seedset
    sig = sigmas_for(steps, srange)
    m0 = res.get(C1.method_tag({"method": "M0", "k": 0, "steps": steps, "seedset": seedset}))
    r = res.get(rule_tag)
    out = {"rule": rule_tag}
    if not m0 or "fid" not in m0 or not r or "fid" not in r or not r.get("complete"):
        out["status"] = "未判定（M0 か規則の FID がまだありません）"
        return out
    tol = (1 + P["fid_rel_tol"]) * m0["fid"]
    out.update({"fid_M0": m0["fid"], "fid_tol": tol, "fid_rule": r["fid"], "cost_rule": r["cost_ratio_mean"],
                "switch_sigmas": r.get("switch_sigmas")})
    out["a_fid_ok"] = r["fid"] <= tol
    out["a_cost_ok"] = r["cost_ratio_mean"] <= P["cost_max"]
    out["a"] = "支持" if (out["a_fid_ok"] and out["a_cost_ok"]) else "不支持"
    grid = sorted((v["k"], v["fid"], v["cost_ratio_mean"]) for v in res.values()
                  if v["method"] == "WV_tele" and v["steps"] == steps and v["seedset"] == seedset
                  and "fid" in v and v.get("complete"))
    out["grid_WV_tele"] = grid
    n_grid = len(C1.wv_k_grid(sig, P["wv_sigma_targets"]))
    if len(grid) < n_grid:
        out["b"] = "未判定（WV_tele の格子が %d/%d 本）" % (len(grid), n_grid)
    else:
        safe = [g for g in grid if g[1] <= tol]
        kb, fb, cb = max(safe, key=lambda z: z[0]) if safe else (0, m0["fid"], 1.0)
        out["WV_tele_best"] = {"k": kb, "fid": fb, "cost": cb, "sigma": float(sig[kb])}
        ok = r["fid"] <= (1 + P["grid_fid_rel"]) * fb and r["cost_ratio_mean"] <= cb + P["grid_cost_margin"]
        out["b_fid_ok"] = r["fid"] <= (1 + P["grid_fid_rel"]) * fb
        out["b_cost_ok"] = r["cost_ratio_mean"] <= cb + P["grid_cost_margin"]
        out["b"] = "支持" if ok else "不支持"
    noise = res.get(C1.method_tag({"method": "M0", "k": 0, "steps": steps, "seedset": P["seedset_noise"]}))
    if noise and "fid" in noise:
        out["fid_M0_other_seeds"] = noise["fid"]
        out["fid_seed_spread"] = abs(noise["fid"] - m0["fid"])
        out["rule_minus_M0"] = r["fid"] - m0["fid"]
        out["within_seed_spread"] = abs(r["fid"] - m0["fid"]) <= abs(noise["fid"] - m0["fid"])
    out["status"] = "判定済み"
    return out


def judge_all(res_vp, res_ve, srange_vp=None, srange_ve=None, P=PREREG2):
    main = tag(umin_jobs(cs=[P["c_min_main"]])[0])
    sub = tag(umin_jobs(cs=[P["c_min_sub"]])[0])
    sgap = C1.method_tag({"method": "S_gap", "k": 0, "steps": P["steps"], "seedset": P["seedset_main"]})
    jve = judge_rule(res_ve, main, srange=srange_ve)
    jvp = judge_rule(res_vp, main, srange=srange_vp)
    J = {"HC5": jve.get("a", jve["status"]), "HC6": jve.get("b", jve["status"]),
         "HC7": jvp.get("a", jvp["status"]), "HC8": jvp.get("b", jvp["status"]),
         "detail_VE_main": jve, "detail_VP_main": jvp,
         "obs_VE_sub": judge_rule(res_ve, sub, srange=srange_ve),
         "obs_VP_sub": judge_rule(res_vp, sub, srange=srange_vp),
         "obs_VE_S_gap": judge_rule(res_ve, sgap, srange=srange_ve),
         "obs_VP_S_gap_C1": judge_rule(res_vp, sgap, srange=srange_vp)}
    return J


# ---------------------------------------------------------------------------
# 確かめ：C2 の U_min の最初の評価（呼び出し 0）が、C1 の S_gap の呼び出し 0 と同じか
# （出発点・5% の点・網・D_G が同じなら、呼び出し 0 の u は同じ。呼び出し 1 からは、
#   C1 の S_gap が 5% の点を網の出力で進め、U_min（B21）は D_G で進めるので、比べない）
# ---------------------------------------------------------------------------
def replication_vs_c1(c1_out, c2_out, steps=None, seedset=None, c_min=None):
    steps = PREREG2["steps"] if steps is None else steps
    seedset = PREREG2["seedset_main"] if seedset is None else seedset
    c_min = PREREG2["c_min_main"] if c_min is None else c_min
    d1 = os.path.join(c1_out, "runs", C1.method_tag({"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}))
    d2 = os.path.join(c2_out, "runs", tag({"method": "U_min", "c_min": c_min, "steps": steps, "seedset": seedset}))
    rows = []
    worst = 0.0
    n_cmp = 0
    if not (os.path.isdir(d1) and os.path.isdir(d2)):
        return {"status": "比べるものがありません", "rows": rows}
    for f in sorted(x for x in os.listdir(d2) if x.endswith(".json")):
        p1 = os.path.join(d1, f)
        if not os.path.exists(p1):
            continue
        with open(p1, encoding="utf-8") as g:
            a = json.load(g)
        with open(os.path.join(d2, f), encoding="utf-8") as g:
            b = json.load(g)
        lim = min(a["switch_call"] if a.get("switch_call") is not None else 10 ** 9,
                  b["switch_call"] if b.get("switch_call") is not None else 10 ** 9)
        ta = {int(r[0]): r for r in a["trace"] if len(r) >= 4}
        tb = {int(r[0]): r for r in b["trace"]}
        cmp_calls = [c for c in sorted(tb) if c < lim and c in ta and c == 0]
        rel = [abs(tb[c][2] - ta[c][3]) / max(abs(ta[c][3]), 1e-300) for c in cmp_calls]
        rel_gap = [abs(tb[c][3] - ta[c][2]) / max(abs(ta[c][2]), 1e-300) for c in cmp_calls]
        w = max(rel + rel_gap) if cmp_calls else float("nan")
        if cmp_calls:
            worst = max(worst, w)
            n_cmp += len(cmp_calls)
        rows.append({"batch": f, "calls_compared": cmp_calls, "max_rel_diff": w})
    ok = n_cmp > 0 and worst <= PREREG2["replication_rel_tol"]
    return {"status": "再現" if ok else ("比べる呼び出しなし" if n_cmp == 0 else "ずれあり"),
            "max_rel_diff": worst, "n_compared": n_cmp, "rows": rows}


# ---------------------------------------------------------------------------
# 図
# ---------------------------------------------------------------------------
def make_figures2(out_dir, outs, res_by, judged, P=PREREG2):
    """outs = {"VP": c2 の vp フォルダ, "VE": ve フォルダ}、res_by = {"VP": res, "VE": res}"""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    files = []
    # 図 1：u の軌跡と切り替えの位置
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, name in zip(axes, ("VP", "VE")):
        o = outs.get(name)
        drawn = False
        for c, col in ((P["c_min_main"], "C3"), (P["c_min_sub"], "C0")):
            d = os.path.join(o, "runs", tag({"method": "U_min", "c_min": c, "steps": P["steps"],
                                               "seedset": P["seedset_main"]})) if o else None
            if not d or not os.path.isdir(d):
                continue
            for i, fj in enumerate(sorted(x for x in os.listdir(d) if x.endswith(".json"))):
                with open(os.path.join(d, fj), encoding="utf-8") as f:
                    inf = json.load(f)
                tr = np.array([r[:3] for r in inf["trace"]], dtype=float)
                if len(tr):
                    ax.loglog(tr[:, 1], tr[:, 2], ".-", color=col, alpha=0.35,
                              label=("u traces, C_MIN = %g" % c) if i == 0 else None)
                    drawn = True
                if inf["switch_sigma"] > 0:
                    ax.axvline(inf["switch_sigma"], color=col, alpha=0.25, ls="--")
        jd = judged.get("detail_%s_main" % name, {})
        if jd.get("WV_tele_best"):
            ax.axvline(jd["WV_tele_best"]["sigma"], color="C2", lw=2, label="sigma of best WV_tele")
        ax.set_title("%s: u = median |D_G - D_net|^2 / d (5%% subset); dashed = switch" % name)
        ax.set_xlabel("sigma")
        ax.invert_xaxis()
        if drawn:
            ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(out_dir, "fig1_u_traces_switch.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    # 図 2：FID と費用
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, name in zip(axes, ("VP", "VE")):
        res = res_by.get(name, {})
        g = sorted((r["cost_ratio_mean"], r["fid"]) for r in res.values()
                   if r["method"] == "WV_tele" and r["seedset"] == P["seedset_main"] and "fid" in r)
        if g:
            ax.plot([a for a, b in g], [b for a, b in g], "s-", color="gray", label="WV_tele(k)")
        for tg, r in res.items():
            if "fid" not in r or r["seedset"] != P["seedset_main"]:
                continue
            if r["method"] == "M0":
                ax.plot(r["cost_ratio_mean"], r["fid"], "ko", ms=9, label="M0")
            elif r["method"] == "S_gap":
                ax.plot(r["cost_ratio_mean"], r["fid"], "m^", ms=9, label="S_gap")
            elif r["method"] == "U_min":
                main = abs(r.get("c_min", 0) - P["c_min_main"]) < 1e-9
                ax.plot(r["cost_ratio_mean"], r["fid"], "r*" if main else "b*", ms=13,
                        label="U_min %g" % r.get("c_min", 0))
        jd = judged.get("detail_%s_main" % name, {})
        if jd.get("fid_tol"):
            ax.axhline(jd["fid_tol"], color="gray", ls=":", label="1.03 x FID(M0)")
        ax.axvline(P["cost_max"], color="gray", ls="--", lw=0.8)
        ax.set_xlabel("net evaluations / M0")
        ax.set_ylabel("FID (10k images)")
        ax.set_title(name)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(out_dir, "fig2_fid_vs_cost.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    return files


def phase_report2(root, c1_out, ref, srange_vp=None, srange_ve=None, extra=None, n_images=None, batch=None,
                  log=print):
    vp = os.path.join(root, "vp")
    ve = os.path.join(root, "ve")
    kw = {"n_images": n_images or PREREG2["n_images"], "batch": batch or PREREG2["batch"]}
    res_vp = {}
    if os.path.isdir(c1_out):
        res_vp.update(C1.collect(c1_out, ref=ref, **kw))
    if os.path.isdir(vp):
        res_vp.update(collect2(vp, ref=ref, **kw))
    res_ve = collect2(ve, ref=ref, **kw) if os.path.isdir(ve) else {}
    J = judge_all(res_vp, res_ve, srange_vp, srange_ve)
    rep = os.path.join(root, "report")
    J["replication_VP_vs_C1"] = replication_vs_c1(c1_out, vp)
    figs = make_figures2(os.path.join(root, "figures"), {"VP": vp, "VE": ve}, {"VP": res_vp, "VE": res_ve}, J)
    summ = {"version": C2_VERSION, "prereg": PREREG2, "results_VP": res_vp, "results_VE": res_ve,
            "judgement": J, "figures": [os.path.basename(f) for f in figs], "extra": extra or {}}
    os.makedirs(rep, exist_ok=True)
    C1._atomic_json(os.path.join(rep, "summary_C2.json"), summ)
    for k in ("HC5", "HC6", "HC7", "HC8"):
        log("%s：%s" % (k, J[k]))
    log("VP の再現（C1 の S_gap の軌跡と）：%s" % J["replication_VP_vs_C1"]["status"])
    return summ


def estimate_minutes(cost_units_list, sec_per_m0_batch=232.0, n_batches=10, factor=1.0):
    """網の評価（M0 = 1 の単位）の合計から、時間（分）を見積もる。"""
    return float(sum(cost_units_list)) * sec_per_m0_batch * n_batches * factor / 60.0


def pack_report2(root, name="C2_report", max_files=100, top="C2_results"):
    return C1.pack_report(root, name=name, max_files=max_files, top=top)


In [ ]:
%%writefile /content/c3core.py
# c3core.py -- 実験場C 段階C3 の骨組み（MIT License, (c) 2026 Taishi Namba）
#
# まだ誰も u の曲線を見ていない網（EDM の AFHQv2 64×64 の VP 型と VE 型。網ごとに別に判定）で、
#   M0、WV_tele の格子（大きい k から、画質を落とさない最初の k で止める）、
#   B21 の U_min（2.38、比較）、B22 の規則（差し込み口）
# を一度だけ走らせるための道具。c1core.py と c2core.py（C1・C2 で Colab が走らせたもの）は import し、手を入れない。
#
# B22 の規則は、定義書 `B22_rule_for_C.md` が届いてから install_b22() で差し込む。
# 差し込むまでは、B22 の仕事を走らせようとすると止まる。

import json
import math
import os
import time

import numpy as np

import c1core as C1
import c2core as C2

C3_VERSION = "C3-1.0"

# ---------------------------------------------------------------------------
# 事前登録の定数（下書き。B22 の欄は定義書が届いてから書き入れる。確定のときに C3_事前登録.md と照合する）
# ---------------------------------------------------------------------------
PREREG3 = {
    "models": {"vp": "edm-afhqv2-64x64-uncond-vp.pkl", "ve": "edm-afhqv2-64x64-uncond-ve.pkl"},
    "hc_names": {"vp": {"U_env": ["HC9", "HC10"], "U_min": ["HC13", "HC14"]},
                 "ve": {"U_env": ["HC11", "HC12"], "U_min": ["HC15", "HC16"]}},
    "gpu": "NVIDIA L4 (both nets)",
    "dataset_n_images": 15803,
    "results_layout": "C3_results/{vp,ve}/ (net-specific); data and moments shared in C3_results/",
    "fid_ref": "afhqv2-64x64.npz",
    "dataset": "afhqv2-64x64 (StarGAN v2 download.sh afhq-v2-dataset → EDM dataset_tool.py --resolution=64x64)",
    "img_shape": [3, 64, 64],
    "steps": 40,
    "sampler_sigma_min": 0.002,
    "sampler_sigma_max": 80.0,
    "sampler_rho": 7.0,
    "batch": 1000,
    "n_images": 5000,
    "seedset_main": 0,
    "seedset_noise": 1,
    "wv_sigma_targets": [40.0, 20.0, 10.0, 5.0, 2.5, 1.25, 0.6],
    "wv_grid_order": "descending k, stop at the first k with FID <= 1.03 x FID(M0)",
    "fid_rel_tol": 0.03,
    "cost_max": 0.85,
    "grid_fid_rel": 0.03,
    "grid_cost_margin": 0.10,
    "umin_c_min": 2.38,
    "subset_frac": 0.05,
    "time_thin": 1,
    "b22_rule": "U_env (B22_rule_for_C.md section 3): W_k = {i <= k : ln sigma_i <= ln sigma_k + DELTA}, extended to the last M_MIN evaluations; "
                "X_i = max(u over W_i); P_k = min(X_0..X_{a_k - 1}); switch at the first k with a_k >= 1 and min(u over W_k) >= C x P_k; "
                "u = median of u_i over finite points; T1P5; before the switch all points advance with D_G",
    "b22_constants": {"DELTA": 0.3, "C": 1.5150000000000001, "M_MIN": 3, "window_tol": 1e-12,
                      "source": "B22/results/rule_constants_B22.json (C = 1.01 x 1.5)"},
    "uenv_window_sizes_40": [1, 2, 3, 4, 5, 5, 6, 5, 6, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
    "uenv_first_decidable_eval_40": 5,
}


def srange3(P=PREREG3):
    return (P["sampler_sigma_min"], P["sampler_sigma_max"])


def sigmas3(steps=None, srange=None, P=PREREG3):
    steps = P["steps"] if steps is None else steps
    lo, hi = srange if srange is not None else srange3(P)
    return C1.edm_sigmas(steps, sigma_min=lo, sigma_max=hi, rho=P["sampler_rho"])


def nfe(steps):
    return C1.nfe_m0(steps)


# ---------------------------------------------------------------------------
# 一般の「合図で切り替える」走らせ方（U_min も B22 もここを通る）
#   rule.observe(sigma, stat) -> True なら、その呼び出しから全部の点を網へ（後戻りしない）
#   stat_fn(DG_sub, Dnet_sub, x_sub, sigma) -> float（既定は B21 の u：|D_G − D_net|²/d の中央値）
#   time_thin = T：切り替える前は、呼び出しの番号が T の倍数のときだけ評価（B21 §4 と同じ数え方）
#   pre_switch = "DG"：切り替える前は全部の点を D_G で進める（B21）
# ---------------------------------------------------------------------------
def stat_u(DGs, Dns, xs, sigma):
    return float(np.median(C2.u_points(DGs, Dns)))


def stat_u_finite(DGs, Dns, xs, sigma):
    """B22 §2：網の出力が有限でない点は除いた u_i の中央値（有限な点がなければ +inf）。"""
    Dn = C1.to_np(Dns)
    fin = np.isfinite(Dn.reshape(Dn.shape[0], -1)).all(1)
    if not fin.any():
        return float("inf")
    if fin.all():
        return float(np.median(C2.u_points(DGs, Dns)))
    idx = C1._idx(np.where(fin)[0], DGs)
    return float(np.median(C2.u_points(DGs[idx], Dns[idx])))


class UEnvRule:
    """B22 の U_env（B22_rule_for_C.md §3 の定義どおり。オンライン）。

    評価 k（0 始まり）の ln σ_k は増えない列。
    1. 窓 W_k = { i ≤ k : ln σ_i ≤ ln σ_k + Δ }（B22 と同じく 1e-12 の余裕で含める）。3 個より少なければ直近 3 個に広げる。
       a_k = 窓の最初の番号。
    2. 上の包絡 X_i = max(u_j : j ∈ W_i)。
    3. 基準 P_k = min(X_0, …, X_{a_k − 1})。
    4. a_k ≥ 1 で min(u_j : j ∈ W_k) ≥ C × P_k となった最初の k で切り替える（後戻りしない）。
    """

    def __init__(self, delta, C, m_min=3, tol=1e-12):
        self.delta, self.C, self.m_min, self.tol = float(delta), float(C), int(m_min), float(tol)
        self.lns, self.neg, self.us, self.X, self.Pmin = [], [], [], [], []
        self.switched = False
        self.k_switch = None
        self.last = None

    def window_start(self, j):
        # B22（common_b22.window_start）と同じ計算：−ln σ の増える列で、−ln σ_k − Δ − tol 以上の最初の番号
        import bisect
        a = bisect.bisect_left(self.neg, self.neg[j] - self.delta - self.tol, 0, j + 1)
        return min(a, max(j - self.m_min + 1, 0))

    def observe(self, sigma, u):
        if self.switched:
            return True
        self.lns.append(math.log(float(sigma)))
        self.neg.append(-self.lns[-1])
        self.us.append(float(u))
        j = len(self.us) - 1
        if j >= 1 and self.lns[j] > self.lns[j - 1] + 1e-15:
            raise ValueError("U_env：ln σ は増えない列でなければなりません")
        a = self.window_start(j)
        w = self.us[a:j + 1]
        self.X.append(max(w))
        self.Pmin.append(self.X[-1] if not self.Pmin else min(self.Pmin[-1], self.X[-1]))
        lo = min(w)
        base = self.Pmin[a - 1] if a >= 1 else None
        self.last = {"k": j, "a": a, "window_n": j - a + 1, "min_w": lo, "P": base}
        if a >= 1 and lo >= self.C * base:
            self.switched = True
            self.k_switch = j
            return True
        return False

    def state(self):
        return {"n_eval": len(self.us), "k_switch": self.k_switch, "last": self.last}


def uenv_window_sizes(sigmas_eval, delta, m_min=3, tol=1e-12):
    """u を使わず、評価の σ の列だけから、各評価の窓の大きさと a_k を出す。"""
    r = UEnvRule(delta, 1.0, m_min, tol)
    out = []
    for s in sigmas_eval:
        r.lns.append(math.log(float(s)))
        r.neg.append(-r.lns[-1])
        j = len(r.lns) - 1
        a = r.window_start(j)
        out.append({"k": j, "sigma": float(s), "a": a, "window_n": j - a + 1, "decidable": a >= 1})
    return out


def eval_sigmas(sigmas, time_thin=1):
    """T1 で評価する呼び出しの σ（Heun：σ_0, σ_1, σ_1, σ_2, σ_2, …）。"""
    return [s for c, (_, s) in enumerate(C1.call_sigmas(sigmas)) if c % int(time_thin) == 0]


def install_uenv(P=PREREG3):
    """B22 の U_env を、事前登録の定数で差し込む。"""
    k = P["b22_constants"]
    install_b22("U_env", lambda: UEnvRule(k["DELTA"], k["C"], k["M_MIN"], k["window_tol"]),
                {"DELTA": k["DELTA"], "C": k["C"], "M_MIN": k["M_MIN"]}, stat_fn=stat_u_finite, time_thin=1,
                frac=P["subset_frac"], source="B22_rule_for_C.md")


class UMinAdapter:
    """B21 の U_min（c2core.UMinRule）を、observe(sigma, stat) の形にする。"""

    def __init__(self, c_min):
        self.r = C2.UMinRule(c_min)

    @property
    def switched(self):
        return self.r.switched

    def observe(self, sigma, stat):
        return self.r.observe(stat)

    def state(self):
        return {"min": self.r.min, "n_eval": self.r.n_eval, "min_at_switch": self.r.min_at_switch}


def run_signal_rule(rule, x_init, sigmas, net_fn, G, rng, stat_fn=stat_u, frac=None, time_thin=1,
                    pre_switch="DG", sync=None):
    frac = PREREG3["subset_frac"] if frac is None else frac
    assert pre_switch == "DG", "この骨組みは B21 の形（切り替える前は全部の点を D_G）だけを用意しています"
    n = int(x_init.shape[0])
    net = C1.CountingNet(net_fn)
    N = len(sigmas) - 1
    m = max(1, int(round(frac * n)))
    trace = []
    st = {"switch_sigma": None, "switch_call": None, "switched": False}
    if sync:
        sync()
    t0 = time.time()

    def D_call(x, s, i, c):
        if st["switched"]:
            return net(x, s)
        DG = G.denoise(x, s)
        if c % int(time_thin) != 0:
            return DG
        sub = np.sort(rng.choice(n, size=m, replace=False))
        isub = C1._idx(sub, x)
        xs = x[isub]
        Dn_sub = net(xs, s)
        DGs = DG[isub]
        val = float(stat_fn(DGs, Dn_sub, xs, float(s)))
        u = float(np.median(C2.u_points(DGs, Dn_sub)))
        sw = bool(rule.observe(float(s), val))
        trace.append([int(c), float(s), val, u])
        if not sw:
            return DG
        out = C1._clone(DG)
        out[isub] = Dn_sub
        rest = np.setdiff1d(np.arange(n), sub)
        if len(rest) > 0:
            irest = C1._idx(rest, x)
            out[irest] = net(x[irest], s)
        st.update(switched=True, switch_sigma=float(s), switch_call=int(c))
        return out

    x = C1.heun_sample(x_init, sigmas, D_call)
    if sync:
        sync()
    info = {"n": n, "steps": N, "subset_m": m, "time_thin": int(time_thin), "trace": trace,
            "trace_columns": ["call", "sigma", "stat", "u"], "switched": st["switched"],
            "switch_sigma": st["switch_sigma"] if st["switched"] else 0.0, "switch_call": st["switch_call"],
            "seconds": time.time() - t0, "net_points": net.points}
    if hasattr(rule, "state"):
        info["rule_state"] = rule.state()
    info["cost_units"] = net.points / n
    info["cost_ratio"] = info["cost_units"] / C1.nfe_m0(N)
    return x, info


# ---------------------------------------------------------------------------
# B22 の差し込み口
# ---------------------------------------------------------------------------
B22 = {"installed": False, "name": None, "make_rule": None, "stat_fn": stat_u, "time_thin": 1,
       "frac": None, "constants": None, "source": None}


def install_b22(name, make_rule, constants, stat_fn=None, time_thin=1, frac=None, source="B22_rule_for_C.md"):
    """定義書どおりの規則を差し込む。make_rule() は observe(sigma, stat) を持つ新しい規則を返す。"""
    B22.update(installed=True, name=name, make_rule=make_rule, constants=constants,
               stat_fn=stat_fn or stat_u, time_thin=int(time_thin), frac=frac, source=source)


def b22_status():
    return {k: (v if k not in ("make_rule", "stat_fn") else (v is not None)) for k, v in B22.items()}


# ---------------------------------------------------------------------------
# 仕事
# ---------------------------------------------------------------------------
def tag3(job):
    if job["method"] == "U_min":
        return C2.tag(job)
    if job["method"] == "B22":
        return "U_env_N%03d_s%d" % (job["steps"], job["seedset"])
    return C1.method_tag(job)


def wv_desc_ks(sigmas, P=PREREG3):
    return sorted(C1.wv_k_grid(sigmas, P["wv_sigma_targets"]), reverse=True)


def run_job3(job, x0, sig, net_fn, G, rng, sync=None):
    if job["method"] == "U_min":
        x, info = run_signal_rule(UMinAdapter(job["c_min"]), x0, sig, net_fn, G, rng, sync=sync)
        info.update({"method": "U_min", "k": 0, "c_min": job["c_min"]})
        return x, info
    if job["method"] == "B22":
        if not B22["installed"]:
            raise RuntimeError("B22 の規則がまだ差し込まれていません（定義書が届いてから install_b22）")
        x, info = run_signal_rule(B22["make_rule"](), x0, sig, net_fn, G, rng, stat_fn=B22["stat_fn"],
                                  frac=B22["frac"], time_thin=B22["time_thin"], sync=sync)
        info.update({"method": "B22", "k": 0, "b22_name": B22["name"], "b22_constants": B22["constants"]})
        return x, info
    return C1.run_one(job["method"], job["k"], x0, sig, net_fn, G, rng=rng, sync=sync)


def phase_runs3(out, jobs, net_fn, G, make_latents, srange=None, n_images=None, batch=None, img_shape=None,
                sync=None, log=print, show_switch=False):
    """1 束 1 ファイル。切り替えの σ は log.txt にだけ書き、画面には出さない（show_switch=False）。"""
    n_images = PREREG3["n_images"] if n_images is None else n_images
    batch = PREREG3["batch"] if batch is None else batch
    img_shape = tuple(PREREG3["img_shape"]) if img_shape is None else img_shape
    nb = int(math.ceil(n_images / batch))
    for job in jobs:
        tg = tag3(job)
        d = os.path.join(out, "runs", tg)
        os.makedirs(d, exist_ok=True)
        sig = sigmas3(job["steps"], srange)
        for b in range(nb):
            fi = os.path.join(d, "b%03d.npz" % b)
            fj = os.path.join(d, "b%03d.json" % b)
            if os.path.exists(fi) and os.path.exists(fj):
                continue
            n = min(batch, n_images - b * batch)
            x0 = make_latents(job["seedset"], b, n) * float(sig[0])
            rng = C1.subset_rng(job["seedset"], job["steps"], b)
            x, info = run_job3(job, x0, sig, net_fn, G, rng, sync=sync)
            info.update({"tag": tg, "batch": b, "seedset": job["seedset"], "version": C3_VERSION,
                         "sigma_range": [float(sig[-2]), float(sig[0])]})
            C1._atomic_npz(fi, images=C1.to_uint8_images(x, img_shape))
            C1._atomic_json(fj, info)
            msg = "%s b%03d  費用 %.3f × M0  %.1f 秒" % (tg, b, info["cost_ratio"], info["seconds"])
            full = msg + ("  切り替え σ = %.4g" % info["switch_sigma"] if "switch_sigma" in info else "")
            log(full if show_switch else msg)
            with open(os.path.join(out, "log.txt"), "a", encoding="utf-8") as f:
                f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + full + "\n")


def fid_of(out, tg, ref, n_images=None):
    n_images = PREREG3["n_images"] if n_images is None else n_images
    fp = os.path.join(out, "runs", tg, "feats_%d.npy" % n_images)
    if not os.path.exists(fp):
        return None
    mu, sg = C1.feat_stats(np.load(fp))
    return C1.fid_from_stats(mu, sg, ref[0], ref[1])


def phase_grid_adaptive(out, net_fn, G, make_latents, feat_fn, ref, srange=None, n_images=None, batch=None,
                        img_shape=None, sync=None, log=print, P=PREREG3):
    """WV_tele を大きい k（安い方）から走らせ、FID ≤ 1.03 × FID(M0) になった最初の k で止める。
    M0（種の組 0）の FID が先に要る。返り値：走らせた k の一覧と、止まった k（なければ None）。"""
    n_images = P["n_images"] if n_images is None else n_images
    batch = P["batch"] if batch is None else batch
    steps, s0 = P["steps"], P["seedset_main"]
    m0 = tag3({"method": "M0", "k": 0, "steps": steps, "seedset": s0})
    f0 = fid_of(out, m0, ref, n_images)
    if f0 is None:
        raise RuntimeError("先に M0 の FID が要ります")
    tol = (1 + P["fid_rel_tol"]) * f0
    sig = sigmas3(steps, srange)
    done, stop = [], None
    for k in wv_desc_ks(sig, P):
        job = {"method": "WV_tele", "k": k, "steps": steps, "seedset": s0}
        phase_runs3(out, [job], net_fn, G, make_latents, srange=srange, n_images=n_images, batch=batch,
                    img_shape=img_shape, sync=sync, log=log)
        C1.phase_features(out, [tag3(job)], feat_fn, n_images=n_images, batch=batch, log=log)
        f = fid_of(out, tag3(job), ref, n_images)
        done.append(k)
        log("WV_tele k=%d（σ = %.3g）：FID %.4f（基準 %.4f）" % (k, sig[k], f, tol))
        if f <= tol:
            stop = k
            break
    C1._atomic_json(os.path.join(out, "grid_state.json"), {"done": done, "stop": stop, "tol": tol, "fid_M0": f0})
    return done, stop


# ---------------------------------------------------------------------------
# 判定（C2 と同じ形。格子は「大きい k から止めるまで」）
# ---------------------------------------------------------------------------
def grid_best(res, sig, steps, seedset, tol, f0, P=PREREG3):
    """走らせた格子から k_best（画質を落とさない最大の k）を決める。
    大きい k から順に、すべて走っていて、最初に基準内になった k が k_best。
    全部の k を走らせて基準内がなければ k_best = 0（M0）。それ以外は未判定。"""
    ks = wv_desc_ks(sig, P)
    have = {v["k"]: v for v in res.values() if v["method"] == "WV_tele" and v["steps"] == steps
            and v["seedset"] == seedset and "fid" in v and v.get("complete")}
    for k in ks:
        if k not in have:
            return None, "未判定（WV_tele k=%d の FID がまだありません）" % k
        if have[k]["fid"] <= tol:
            return {"k": k, "fid": have[k]["fid"], "cost": have[k]["cost_ratio_mean"], "sigma": float(sig[k])}, None
    return {"k": 0, "fid": f0, "cost": 1.0, "sigma": float(sig[0])}, None


def judge3(res, rule_tag, P=PREREG3, srange=None):
    steps, s0 = P["steps"], P["seedset_main"]
    sig = sigmas3(steps, srange)
    m0 = res.get(tag3({"method": "M0", "k": 0, "steps": steps, "seedset": s0}))
    r = res.get(rule_tag)
    out = {"rule": rule_tag}
    if not m0 or "fid" not in m0 or not r or "fid" not in r or not r.get("complete"):
        out["status"] = "未判定（M0 か規則の FID がまだありません）"
        out["a"] = out["b"] = out["status"]
        return out
    tol = (1 + P["fid_rel_tol"]) * m0["fid"]
    out.update({"fid_M0": m0["fid"], "fid_tol": tol, "fid_rule": r["fid"], "cost_rule": r["cost_ratio_mean"],
                "switch_sigmas": r.get("switch_sigmas")})
    out["a"] = "支持" if (r["fid"] <= tol and r["cost_ratio_mean"] <= P["cost_max"]) else "不支持"
    best, why = grid_best(res, sig, steps, s0, tol, m0["fid"], P)
    if best is None:
        out["b"] = why
    else:
        out["WV_tele_best"] = best
        ok = r["fid"] <= (1 + P["grid_fid_rel"]) * best["fid"] and r["cost_ratio_mean"] <= best["cost"] + P["grid_cost_margin"]
        out["b"] = "支持" if ok else "不支持"
    noise = res.get(tag3({"method": "M0", "k": 0, "steps": steps, "seedset": P["seedset_noise"]}))
    if noise and "fid" in noise:
        out["fid_M0_other_seeds"] = noise["fid"]
        out["fid_seed_spread"] = abs(noise["fid"] - m0["fid"])
        out["within_seed_spread"] = abs(r["fid"] - m0["fid"]) <= out["fid_seed_spread"]
    out["status"] = "判定済み"
    return out


def collect3(out, ref=None, n_images=None, batch=None, P=PREREG3):
    n_images = P["n_images"] if n_images is None else n_images
    batch = P["batch"] if batch is None else batch
    res = C1.collect(out, n_images=n_images, batch=batch, ref=ref)
    for tg, r in res.items():
        if r["method"] in ("U_min", "B22"):
            with open(os.path.join(out, "runs", tg, "b000.json"), encoding="utf-8") as f:
                j = json.load(f)
            r["c_min"] = j.get("c_min")
            r["b22_name"] = j.get("b22_name")
    return res


def judge_all3(res, srange=None, P=PREREG3, net=None):
    """二つの規則（U_env = B22、U_min 2.38 = B21）を、どちらも判定する。net を渡すと HC 番号を付ける。"""
    umin = tag3({"method": "U_min", "c_min": P["umin_c_min"], "steps": P["steps"], "seedset": P["seedset_main"]})
    b22 = tag3({"method": "B22", "steps": P["steps"], "seedset": P["seedset_main"]})
    jb = judge3(res, b22, P, srange)
    ju = judge3(res, umin, P, srange)
    # HC 番号は、B22 の定義書が届いて事前登録を確定するときに決める（下書きでは仮の名前）
    J = {"U_env_a": jb["a"], "U_env_b": jb["b"], "detail_B22": jb,
         "U_min_a": ju["a"], "U_min_b": ju["b"], "detail_U_min": ju}
    if net is not None:
        ea, eb = P["hc_names"][net]["U_env"]
        ma, mb = P["hc_names"][net]["U_min"]
        J.update({"net": net, ea: jb["a"], eb: jb["b"], ma: ju["a"], mb: ju["b"]})
    return J


def report3(out, ref, srange=None, n_images=None, batch=None, extra=None, log=print, P=PREREG3, net=None):
    res = collect3(out, ref=ref, n_images=n_images, batch=batch, P=P)
    J = judge_all3(res, srange, P, net=net)
    rep = os.path.join(out, "report")
    os.makedirs(rep, exist_ok=True)
    figs = figures3(out, res, J, P)
    summ = {"version": C3_VERSION, "prereg": P, "b22": b22_status(), "results": res, "judgement": J,
            "figures": [os.path.basename(f) for f in figs], "extra": extra or {}}
    C1._atomic_json(os.path.join(rep, "summary_C3.json"), summ)
    keys = (P["hc_names"][net]["U_env"] + P["hc_names"][net]["U_min"]) if net else ["U_env_a", "U_env_b", "U_min_a", "U_min_b"]
    for k in keys:
        log("%s%s：%s" % ("[%s] " % net if net else "", k, J[k]))
    return summ


def figures3(out, res, J, P=PREREG3):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fd = os.path.join(out, "figures")
    os.makedirs(fd, exist_ok=True)
    files = []
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for tg, col in ((tag3({"method": "U_min", "c_min": P["umin_c_min"], "steps": P["steps"], "seedset": 0}), "C3"),
                    (tag3({"method": "B22", "steps": P["steps"], "seedset": 0}), "C0")):
        d = os.path.join(out, "runs", tg)
        if not os.path.isdir(d):
            continue
        for i, fj in enumerate(sorted(x for x in os.listdir(d) if x.endswith(".json"))):
            with open(os.path.join(d, fj), encoding="utf-8") as f:
                inf = json.load(f)
            tr = np.array([[r[1], r[3]] for r in inf["trace"]], dtype=float)
            if len(tr):
                ax.loglog(tr[:, 0], tr[:, 1], ".-", color=col, alpha=0.35, label=tg if i == 0 else None)
            if inf.get("switch_sigma", 0) > 0:
                ax.axvline(inf["switch_sigma"], color=col, ls="--", alpha=0.25)
    best = J["detail_B22"].get("WV_tele_best") or J["detail_U_min"].get("WV_tele_best")
    if best:
        ax.axvline(best["sigma"], color="C2", lw=2, label="sigma of best WV_tele")
    ax.invert_xaxis()
    ax.set_xlabel("sigma")
    ax.set_title("u = median |D_G - D_net|^2 / d (5%% subset); dashed = switch")
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(fd, "fig1_u_traces_switch.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    g = sorted((r["cost_ratio_mean"], r["fid"]) for r in res.values() if r["method"] == "WV_tele" and "fid" in r)
    if g:
        ax.plot([a for a, b in g], [b for a, b in g], "s-", color="gray", label="WV_tele(k) run")
    for tg, r in res.items():
        if "fid" not in r or r["seedset"] != P["seedset_main"]:
            continue
        mk = {"M0": "ko", "U_min": "r*", "B22": "b*"}.get(r["method"])
        if mk:
            ax.plot(r["cost_ratio_mean"], r["fid"], mk, ms=11, label=tg)
    jd = J["detail_B22"] if "fid_tol" in J["detail_B22"] else J["detail_U_min"]
    if jd.get("fid_tol"):
        ax.axhline(jd["fid_tol"], color="gray", ls=":", label="1.03 x FID(M0)")
    ax.axvline(P["cost_max"], color="gray", ls="--", lw=0.8)
    ax.set_xlabel("net evaluations / M0")
    ax.set_ylabel("FID (%d images)" % P["n_images"])
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(fd, "fig2_fid_vs_cost.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    return files


def estimate_hours(ratio_vs_cifar=1.8, steps=40, n_images=5000, units=6.0, cifar_sec_per_1000_m0=232.0):
    """CIFAR の M0（18 歩、35 回）の実測から、64×64・steps 歩での M0 1 本の時間と、全体の時間を見積もる。"""
    per_img_nfe = cifar_sec_per_1000_m0 / 1000 / 35
    m0_hours = per_img_nfe * ratio_vs_cifar * C1.nfe_m0(steps) * n_images / 3600
    return {"M0_hours": m0_hours, "total_hours": m0_hours * units}


def pack_report3(out, name="C3_report", max_files=100, top="C3_results"):
    return C1.pack_report(out, name=name, max_files=max_files, top=top)


# ---------------------------------------------------------------------------
# 網ごとのフォルダ（C3b）
#   C3_results/                      データと積率（二つの網で共通）：afhqv2_64_moments.npz、fid_real_5k.json
#   C3_results/vp/  C3_results/ve/   網ごとの結果：runs/、log.txt、net_info.json、grid_state.json、report/、figures/
# C3（版 C3-0.1）のノートブックは、VP 型の結果を C3_results/ の直下に書いていた。
# migrate_legacy() は、それを C3_results/vp/ へ名前の付け替え（os.replace）だけで移す。計算し直さない。
# ---------------------------------------------------------------------------
NETS = ("vp", "ve")
SHARED_FILES = ("afhqv2_64_moments.npz", "fid_real_5k.json")
LEGACY_NET_ITEMS = ("runs", "log.txt", "net_info.json", "grid_state.json", "report", "figures")


def net_dir(root, net):
    assert net in NETS, "NET は 'vp' か 've'"
    return os.path.join(root, net)


def moments_path(root):
    """s_G の積率は、網によらず一つ（データだけで決まる）。"""
    return os.path.join(root, SHARED_FILES[0])


def net_url(net, base="https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/", P=PREREG3):
    return base + P["models"][net]


def migrate_legacy(root, log=print):
    """C3_results/ の直下にある VP 型の結果（C3 の版）を、C3_results/vp/ へ移す。
    何度呼んでも同じ（済んだものは飛ばす）。同じ名前が両方にあれば、上書きせずに止まる。"""
    vp = net_dir(root, "vp")
    moved, conflicts = [], []
    for it in LEGACY_NET_ITEMS:
        src, dst = os.path.join(root, it), os.path.join(vp, it)
        if it == "report" and os.path.exists(os.path.join(src, "summary_C3_both.json")):
            continue  # C4 の直し：直下の report/ に二つの網をまとめた集計があれば、それは C3b 以後の集計（VP 型の結果ではない）
        if os.path.exists(src):
            if os.path.exists(dst):
                conflicts.append(it)
                continue
            os.makedirs(vp, exist_ok=True)
            os.replace(src, dst)
            moved.append(it)
    if conflicts:
        raise RuntimeError("C3_results/ の直下と vp/ の両方に %s があります。上書きせずに止まりました。本筋に相談してください。" % conflicts)
    if moved:
        log("C3 の版の VP 型の結果を vp/ へ移しました：%s" % moved)
        with open(os.path.join(vp, "log.txt"), "a", encoding="utf-8") as f:
            f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + "MIGRATED from C3_results/ (C3-0.1 layout): %s\n" % moved)
    return moved


def report_both(root, ref, srange=None, n_images=None, batch=None, extras=None, log=print, P=PREREG3):
    """二つの網を、それぞれ別に判定して並べる（どちらかが支持なら良い、という読み方はしない）。"""
    both = {"version": C3_VERSION, "nets": {}}
    for net in NETS:
        d = net_dir(root, net)
        if not os.path.isdir(os.path.join(d, "runs")):
            both["nets"][net] = {"status": "結果がまだありません"}
            continue
        summ = report3(d, ref, srange=srange, n_images=n_images, batch=batch,
                       extra=(extras or {}).get(net), log=log, P=P, net=net)
        J = summ["judgement"]
        both["nets"][net] = {k: J[k] for k in P["hc_names"][net]["U_env"] + P["hc_names"][net]["U_min"]}
    rep = os.path.join(root, "report")
    os.makedirs(rep, exist_ok=True)
    C1._atomic_json(os.path.join(rep, "summary_C3_both.json"), both)
    return both


def all_done(root, P=PREREG3):
    """二つの網の本番の計算（M0・格子・U_min・U_env・別の種の M0 と、その FID の特徴量）が全部そろったか。
    そろうまでは、最後のセルは集計も図も出さない（u の曲線を誰も見ないため）。"""
    steps, s0, s1, n = P["steps"], P["seedset_main"], P["seedset_noise"], P["n_images"]
    need_jobs = [{"method": "M0", "k": 0, "steps": steps, "seedset": s0},
                 {"method": "M0", "k": 0, "steps": steps, "seedset": s1},
                 {"method": "U_min", "k": 0, "c_min": P["umin_c_min"], "steps": steps, "seedset": s0},
                 {"method": "B22", "k": 0, "steps": steps, "seedset": s0}]
    missing = []
    for net in NETS:
        d = net_dir(root, net)
        for j in need_jobs:
            if not os.path.exists(os.path.join(d, "runs", tag3(j), "feats_%d.npy" % n)):
                missing.append("%s/%s" % (net, tag3(j)))
        gs = os.path.join(d, "grid_state.json")
        if not os.path.exists(gs):
            missing.append("%s/grid_state.json" % net)
        else:
            with open(gs, encoding="utf-8") as f:
                g = json.load(f)
            ks = wv_desc_ks(sigmas3(steps, None, P), P)
            if g.get("stop") is None and list(g.get("done", [])) != ks:
                missing.append("%s/格子（途中）" % net)
    return (not missing), missing


In [ ]:
%%writefile /content/c6core.py
# c6core.py -- 実験場C 段階C6：本番（C3）の結果を強める追加の試験（MIT License, (c) 2026 Taishi Namba）
#   A：種の組 2・3 を足す（二つの網、M0・WV_tele(k_best)・U_min 2.38・U_env）
#   B：VE 型の格子を細かくする（WV_tele の k = 10・11・12、種の組 0）
# 規則と走らせ方は c3core.py（版 C3-1.0、C3 の本番で Colab が走らせたもの）を一字も変えずに import して使う。
# C3 の結果（C3_results/）は読むだけで、書き換えない。データと積率も C3 のものを読むだけ。

import json
import math
import os

import numpy as np

import c1core as C1
import c3core as C3

C6_VERSION = "C6-1.0"

PREREG6 = {
    "base_code": "c3core.py C3-1.0 (unchanged)",
    "nets": ["vp", "ve"],
    "steps": 40,
    "n_images_per_seedset": 5000,
    "batch": 1000,
    "seedset_c3_main": 0,
    "seedset_c3_noise": 1,
    "seedsets_new": [2, 3],
    "seed_rule": "latents of batch b in seed set s: torch.Generator(cuda).manual_seed(1000003*s + b), b = 0..4; "
                 "seed sets 0..3 never share a seed because b < 1000003",
    "k_best_c3": {"vp": 13, "ve": 9},
    "grid_rule_A": "k_best is fixed from C3 (seed set 0) and not searched again; WV_tele(k_best) is run on each new seed set "
                   "so that the grid comparison uses FIDs from the same seed set",
    "jobs_A_per_seedset": ["M0", "WV_tele(k_best)", "U_min 2.38", "U_env"],
    "jobs_B": {"net": "ve", "seedset": 0, "k": [10, 11, 12]},
    "fid_rel_tol": 0.03,
    "cost_max": 0.85,
    "grid_fid_rel": 0.03,
    "grid_cost_margin": 0.10,
    "pooled_main": [0, 2, 3],
    "pooled_new_only": [2, 3],
    "hc_pooled_main": {"vp": {"U_env": ["HC17", "HC18"], "U_min": ["HC21", "HC22"]},
                       "ve": {"U_env": ["HC19", "HC20"], "U_min": ["HC23", "HC24"]}},
    "hc_B": "HC25",
}

NB = 5
SIG = C3.sigmas3()


def tag(method, s, k=0):
    if method == "U_min":
        return C3.tag3({"method": "U_min", "c_min": 2.38, "steps": 40, "seedset": s})
    if method == "U_env":
        return C3.tag3({"method": "B22", "steps": 40, "seedset": s})
    return C3.tag3({"method": method, "k": k, "steps": 40, "seedset": s})


def jobs_A(net, P=PREREG6):
    kb = P["k_best_c3"][net]
    J = []
    for s in P["seedsets_new"]:
        J += [{"method": "M0", "k": 0, "steps": 40, "seedset": s},
              {"method": "WV_tele", "k": kb, "steps": 40, "seedset": s},
              {"method": "U_min", "k": 0, "c_min": 2.38, "steps": 40, "seedset": s},
              {"method": "B22", "k": 0, "steps": 40, "seedset": s}]
    return J


def jobs_B(P=PREREG6):
    b = P["jobs_B"]
    return [{"method": "WV_tele", "k": k, "steps": 40, "seedset": b["seedset"]} for k in b["k"]]


def cost_of(method, k=0):
    if method == "WV_tele":
        return (2 * (40 - k) - 1) / 79
    return 1.0


def estimate_hours(net, sec_m0_batch, P=PREREG6, rule_cost=(0.75, 0.75), sec_rule_batch=None):
    """L4 の実測（M0 1 束の秒）から、A と B の時間を見積もる。規則は C3 の実測（1 束 508〜517 秒）を使う。"""
    kb = P["k_best_c3"][net]
    rule = sec_rule_batch if sec_rule_batch else 515.0
    per_seed = NB * (sec_m0_batch + cost_of("WV_tele", kb) * sec_m0_batch + 2 * rule) / 3600
    A = per_seed * len(P["seedsets_new"])
    B = NB * sum(cost_of("WV_tele", k) * sec_m0_batch for k in P["jobs_B"]["k"]) / 3600 if net == "ve" else 0.0
    return {"A_hours": A, "B_hours": B}


# ---------------------------------------------------------------------------
# 特徴量と FID（種の組 0 は C3 の結果から読み、2・3 は C6 の結果から読む）
# ---------------------------------------------------------------------------
def feats_path(root3, root6, net, method, s, k=0, n=5000):
    base = root3 if s in (0, 1) and (method != "WV_tele" or k in (9, 13, 16, 20, 22, 25)) else root6
    return os.path.join(C3.net_dir(base, net), "runs", tag(method, s, k), "feats_%d.npy" % n)


def load_feats(root3, root6, net, method, seedsets, k=0):
    fs = []
    for s in seedsets:
        p = feats_path(root3, root6, net, method, s, k)
        if not os.path.exists(p):
            return None
        fs.append(np.load(p))
    return np.concatenate(fs, 0)


def fid_pooled(root3, root6, net, method, seedsets, ref, k=0):
    f = load_feats(root3, root6, net, method, seedsets, k)
    if f is None:
        return None
    mu, sg = C1.feat_stats(f)
    return C1.fid_from_stats(mu, sg, ref[0], ref[1])


def cost_pooled(root3, root6, net, method, seedsets, k=0):
    cs = []
    for s in seedsets:
        base = root3 if s in (0, 1) else root6
        d = os.path.join(C3.net_dir(base, net), "runs", tag(method, s, k))
        for b in range(NB):
            p = os.path.join(d, "b%03d.json" % b)
            if not os.path.exists(p):
                return None
            with open(p, encoding="utf-8") as f:
                cs.append(json.load(f)["cost_ratio"])
    return float(np.mean(cs))


def judge_one(fid_rule, cost_rule, fid_m0, fid_grid, cost_grid, P=PREREG6):
    """C3 と同じ形の二つの判定。"""
    if None in (fid_rule, cost_rule, fid_m0):
        return {"a": "未判定", "b": "未判定"}
    a = "支持" if (fid_rule <= (1 + P["fid_rel_tol"]) * fid_m0 and cost_rule <= P["cost_max"]) else "不支持"
    if None in (fid_grid, cost_grid):
        b = "未判定"
    else:
        b = "支持" if (fid_rule <= (1 + P["grid_fid_rel"]) * fid_grid and cost_rule <= cost_grid + P["grid_cost_margin"]) else "不支持"
    return {"a": a, "b": b, "fid_rule": fid_rule, "cost_rule": cost_rule, "fid_M0": fid_m0,
            "fid_tol": (1 + P["fid_rel_tol"]) * fid_m0, "fid_grid": fid_grid, "cost_grid": cost_grid,
            "margin_fid_to_tol": (1 + P["fid_rel_tol"]) * fid_m0 - fid_rule,
            "margin_cost_to_grid": None if cost_grid is None else cost_grid + P["grid_cost_margin"] - cost_rule}


def judge6(root3, root6, ref, P=PREREG6):
    out = {"version": C6_VERSION, "nets": {}}
    for net in P["nets"]:
        kb = P["k_best_c3"][net]
        o = {"per_seedset": {}, "pooled_main": {}, "pooled_new_only": {}}
        for s in [0] + P["seedsets_new"]:
            fm0 = fid_pooled(root3, root6, net, "M0", [s], ref)
            fg = fid_pooled(root3, root6, net, "WV_tele", [s], ref, kb)
            cg = cost_of("WV_tele", kb)
            o["per_seedset"][s] = {r: judge_one(fid_pooled(root3, root6, net, r, [s], ref),
                                                cost_pooled(root3, root6, net, r, [s]), fm0, fg, cg, P)
                                   for r in ("U_env", "U_min")}
        for key in ("pooled_main", "pooled_new_only"):
            ss = P[key]
            fm0 = fid_pooled(root3, root6, net, "M0", ss, ref)
            fg = fid_pooled(root3, root6, net, "WV_tele", ss, ref, kb)
            o[key] = {r: judge_one(fid_pooled(root3, root6, net, r, ss, ref), cost_pooled(root3, root6, net, r, ss),
                                   fm0, fg, cost_of("WV_tele", kb), P) for r in ("U_env", "U_min")}
            o[key]["n_images"] = 5000 * len(ss)
        hc = P["hc_pooled_main"][net]
        for r in ("U_env", "U_min"):
            o[hc[r][0]] = o["pooled_main"][r]["a"]
            o[hc[r][1]] = o["pooled_main"][r]["b"]
            per = [o["per_seedset"][s][r] for s in [0] + P["seedsets_new"]]
            o["all_three_%s_a" % r] = ("支持" if all(x["a"] == "支持" for x in per)
                                      else ("未判定" if any(x["a"] == "未判定" for x in per) else "不支持"))
            o["all_three_%s_b" % r] = ("支持" if all(x["b"] == "支持" for x in per)
                                      else ("未判定" if any(x["b"] == "未判定" for x in per) else "不支持"))
        # 種の揺れ（M0 の種の組 0〜3 の FID の幅）
        fm = [fid_pooled(root3, root6, net, "M0", [s], ref) for s in (0, 1, 2, 3)]
        o["fid_M0_by_seedset"] = fm
        if all(x is not None for x in fm):
            o["fid_M0_spread"] = max(fm) - min(fm)
        out["nets"][net] = o
    # B：VE 型の細かい格子（種の組 0）
    fm0 = fid_pooled(root3, root6, "ve", "M0", [0], ref)
    bb = {}
    for k in P["jobs_B"]["k"]:
        f = fid_pooled(root3, root6, "ve", "WV_tele", [0], ref, k)
        bb[k] = {"fid": f, "cost": cost_of("WV_tele", k), "sigma": float(SIG[k]),
                 "within": None if (f is None or fm0 is None) else bool(f <= (1 + P["fid_rel_tol"]) * fm0)}
    if any(v["within"] is None for v in bb.values()):
        hb = "未判定"
    else:
        hb = "支持" if any(v["within"] for v in bb.values()) else "不支持"
    safe = [k for k, v in bb.items() if v["within"]]
    out["B"] = {"grid": bb, "fid_M0_s0": fm0, "tol": None if fm0 is None else (1 + P["fid_rel_tol"]) * fm0,
                P["hc_B"]: hb, "k_best_fine": max(safe) if safe else None}
    return out


# ---------------------------------------------------------------------------
# そろったか（最後のセルは、そろうまで集計も図も出さない）
# ---------------------------------------------------------------------------
def all_done6(root3, root6, P=PREREG6, nets=None):
    nets = P["nets"] if nets is None else nets
    miss = []
    for net in nets:
        for j in jobs_A(net, P):
            m = {"B22": "U_env"}.get(j["method"], j["method"])
            p = feats_path(root3, root6, net, m, j["seedset"], j["k"])
            if not os.path.exists(p):
                miss.append("%s/%s" % (net, tag(m, j["seedset"], j["k"])))
    if "ve" in nets:
        for j in jobs_B(P):
            p = feats_path(root3, root6, "ve", "WV_tele", 0, j["k"])
            if not os.path.exists(p):
                miss.append("ve/%s" % tag("WV_tele", 0, j["k"]))
    return (not miss), miss


def figures6(root3, root6, J, out_dir, P=PREREG6):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    files = []
    # 図 1：種の組ごとの u の列と切り替え（U_env・U_min）
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
    cols = {0: "gray", 2: "C0", 3: "C1"}
    for ax, net in zip(axes, P["nets"]):
        for s in [0] + P["seedsets_new"]:
            base = root3 if s == 0 else root6
            for r, mk in (("U_min", "x"), ("U_env", "+")):
                d = os.path.join(C3.net_dir(base, net), "runs", tag(r, s))
                for b in range(NB):
                    p = os.path.join(d, "b%03d.json" % b)
                    if not os.path.exists(p):
                        continue
                    with open(p, encoding="utf-8") as f:
                        inf = json.load(f)
                    tr = np.array([[x[1], x[3]] for x in inf["trace"]], dtype=float)
                    if r == "U_min":
                        ax.loglog(tr[:, 0], tr[:, 1], ".-", color=cols[s], alpha=0.35, lw=0.8,
                                  label=("seed set %d" % s) if b == 0 else None)
                    ax.plot(tr[-1, 0], tr[-1, 1], mk, color="r" if r == "U_min" else "b", ms=8)
        ax.axvline(SIG[P["k_best_c3"][net]], color="C2", lw=2, alpha=0.5, label="k_best (C3)")
        ax.invert_xaxis()
        ax.set_xlabel("sigma")
        ax.set_title("%s: u traces; x = U_min switch, + = U_env switch" % net.upper())
        ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(out_dir, "figC6_1_u_by_seedset.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    # 図 2：種の組ごとの FID − FID(M0) と、合わせた値
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
    for ax, net in zip(axes, P["nets"]):
        o = J["nets"][net]
        xs = [0] + P["seedsets_new"]
        for i, r in enumerate(("U_env", "U_min")):
            y = [o["per_seedset"][s][r].get("fid_rule", np.nan) - o["per_seedset"][s][r].get("fid_M0", np.nan) for s in xs]
            tol = [0.03 * o["per_seedset"][s][r].get("fid_M0", np.nan) for s in xs]
            ax.plot(np.arange(3) + 0.1 * i, y, "o", label=r)
            if i == 0:
                ax.plot(np.arange(3), tol, "k_", ms=20, label="tolerance (0.03 x FID(M0))")
            pm = o["pooled_main"][r]
            if pm.get("fid_rule") is not None:
                ax.plot(3.2 + 0.1 * i, pm["fid_rule"] - pm["fid_M0"], "s", label="%s pooled 0+2+3" % r)
        ax.set_xticks([0, 1, 2, 3.2])
        ax.set_xticklabels(["seed 0", "seed 2", "seed 3", "pooled"])
        ax.axhline(0, color="gray", lw=0.5)
        ax.set_ylabel("FID(rule) - FID(M0)")
        ax.set_title(net.upper())
        ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(out_dir, "figC6_2_fid_by_seedset.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    return files


def report6(root3, root6, ref, extra=None, log=print, P=PREREG6):
    J = judge6(root3, root6, ref, P)
    figs = figures6(root3, root6, J, os.path.join(root6, "figures"), P)
    rep = os.path.join(root6, "report")
    os.makedirs(rep, exist_ok=True)
    summ = {"version": C6_VERSION, "prereg": P, "judgement": J, "figures": [os.path.basename(f) for f in figs],
            "extra": extra or {}}
    C1._atomic_json(os.path.join(rep, "summary_C6.json"), summ)
    for net in P["nets"]:
        hc = P["hc_pooled_main"][net]
        log("%s：%s" % (net, {k: J["nets"][net][k] for r in ("U_env", "U_min") for k in hc[r]}))
    log("B（%s）：%s" % (P["hc_B"], J["B"][P["hc_B"]]))
    return summ


def pack6(root6, name="C6_report", top="C6_results"):
    return C1.pack_report(root6, name=name, max_files=100, top=top)


In [ ]:
# ⑥ 読み込み（規則は C3-1.0 のまま）。C6 の結果は ROOT6/NET に書く
sys.path.insert(0, '/content')
import importlib, c1core as C1, c2core as C2, c3core as C3, c6core as C6
for m in (C1, C2, C3, C6): importlib.reload(m)
assert C3.C3_VERSION == 'C3-1.0', 'c3core の版が C3-1.0 ではありません'
P = C3.PREREG3
P6 = C6.PREREG6
OUT = C3.net_dir(ROOT6, NET)
os.makedirs(OUT, exist_ok=True)
NET_URL = C3.net_url(NET, EDM_BASE + 'pretrained/')
# C3 の k_best が、C3 の格子の記録と同じかを確かめる
g = json.load(open(os.path.join(C3.net_dir(ROOT3, NET), 'grid_state.json')))
assert g['stop'] == P6['k_best_c3'][NET], 'C3 の k_best が登録と違います。止まって本筋に相談してください。'
print(C3.C3_VERSION, C6.C6_VERSION, '  網：', NET, NET_URL, '\n書き出し先：', OUT, '  C3 の k_best：', g['stop'])


In [ ]:
# ⑦ s_G：C3 で作った積率を読むだけ（作り直さない）。画像の枚数が 15,803 でなければ止まる
MOM = C3.moments_path(ROOT3)
z = np.load(MOM)
print('積率：', MOM, '  画像の枚数：', int(z['n']))
assert int(z['n']) == P['dataset_n_images'], '画像の枚数が登録と違います。止まって本筋に相談してください。'
to_dev = lambda a: torch.as_tensor(a, dtype=torch.float64, device=dev)
G = C1.GaussianModel(z['mu'], z['lam'], z['U'], to_backend=to_dev)
del z; gc.collect()


In [ ]:
# ⑧ 網と FID の道具（C3-1.0 のノートブックと同じ）。出発点の乱数も C3 と同じ形
with dnnlib.util.open_url(NET_URL) as f:
    net = pickle.load(f)['ema'].to(dev).eval()
NET_INFO = {'url': NET_URL, 'class': type(net).__name__, 'img_resolution': int(net.img_resolution),
            'sigma_min': float(net.sigma_min), 'sigma_max': float(net.sigma_max),
            'sigma_data': float(getattr(net, 'sigma_data', float('nan')))}
assert NET_INFO['img_resolution'] == 64
srange = C2.sampler_range(net.sigma_min, net.sigma_max)
sig = C3.sigmas3(P['steps'], srange)
rs = net.round_sigma(torch.as_tensor(sig[:-1], dtype=torch.float64)).cpu().numpy()
NET_INFO.update(sampler_sigma_range=list(srange), round_sigma_max_change=float(np.max(np.abs(rs - sig[:-1]))))
assert NET_INFO['round_sigma_max_change'] < 1e-12 and abs(srange[0] - 0.002) < 1e-12 and abs(srange[1] - 80) < 1e-12, '目盛りが登録と違います。止まって本筋に相談してください。'
NET_INFO['net'] = NET
json.dump(NET_INFO, open(os.path.join(OUT, 'net_info.json'), 'w'), indent=1)

@torch.no_grad()
def net_fn(x, sigma):
    out = torch.empty_like(x)
    s = torch.as_tensor(sigma, dtype=torch.float64, device=dev)
    for i in range(0, x.shape[0], NET_CHUNK):
        xb = x[i:i + NET_CHUNK].reshape(-1, 3, 64, 64)
        out[i:i + NET_CHUNK] = net(xb, s).to(torch.float64).reshape(xb.shape[0], -1)
    return out

def make_latents(seedset, b, n):   # C1〜C3 と同じ形
    g = torch.Generator(device=dev).manual_seed(1000003 * int(seedset) + int(b))
    return torch.randn(n, 12288, generator=g, device=dev, dtype=torch.float64)

sync = torch.cuda.synchronize
with dnnlib.util.open_url(DETECTOR_URL) as f:
    detector = pickle.load(f).to(dev)
with dnnlib.util.open_url(REF_URL) as f:
    refz = dict(np.load(f))
ref = (refz['mu'], refz['sigma'])

@torch.no_grad()
def feat_fn(imgs_u8):
    fs = []
    for i in range(0, len(imgs_u8), 250):
        fs.append(detector(torch.as_tensor(imgs_u8[i:i + 250], device=dev), return_features=True).to(torch.float64).cpu().numpy())
    return np.concatenate(fs)

C3.install_uenv()   # U_env（B22）の定数は C3 の事前登録のまま
print(json.dumps(NET_INFO), '\nU_env：', C3.b22_status()['constants'])


In [ ]:
# ⑨ B：VE 型の WV_tele k = 10・11・12（種の組 0）。FID は、ここでは表示しない
if RUN_B and NET == 've':
    jb = C6.jobs_B()
    with torch.no_grad():
        C3.phase_runs3(OUT, jb[:1], net_fn, G, make_latents, srange=srange, n_images=1000, sync=sync)
    sec = json.load(open(os.path.join(OUT, 'runs', C3.tag3(jb[0]), 'b000.json')))['seconds']
    print('WV_tele k=10 の 1 束：%.0f 秒 → B 全体の見込み 約 %.1f 時間' % (sec, sec * 5 * 3 * 0.93 / 3600))
    with torch.no_grad():
        C3.phase_runs3(OUT, jb, net_fn, G, make_latents, srange=srange, sync=sync)
    C1.phase_features(OUT, [C3.tag3(j) for j in jb], feat_fn, n_images=5000)
elif RUN_B:
    print('B は VE 型だけです（NET = "ve" で走らせてください）')


In [ ]:
# ⑩ A：種の組 2・3（M0、WV_tele(k_best)、U_min 2.38、U_env）。費用と時間だけを表示する
if RUN_A:
    ja = C6.jobs_A(NET)
    with torch.no_grad():
        C3.phase_runs3(OUT, ja[:1], net_fn, G, make_latents, srange=srange, n_images=1000, sync=sync)
    sec = json.load(open(os.path.join(OUT, 'runs', C3.tag3(ja[0]), 'b000.json')))['seconds']
    est = C6.estimate_hours(NET, sec)
    print('M0 の 1 束：%.0f 秒 → A（この網）の見込み 約 %.1f 時間' % (sec, est['A_hours']))
    for s in P6['seedsets_new']:
        js = [j for j in ja if j['seedset'] == s]
        with torch.no_grad():
            C3.phase_runs3(OUT, js, net_fn, G, make_latents, srange=srange, sync=sync)
        C1.phase_features(OUT, [C3.tag3(j) for j in js], feat_fn, n_images=5000)
        print('種の組 %d：済み' % s)


In [ ]:
# ⑪ 集計・判定・図・zip（u の曲線と B の FID は、ここで初めて表示する）
#    A（二つの網）と B がすべてそろうまでは、集計も図も出さずに止まる
done, missing = C6.all_done6(ROOT3, ROOT6)
if not done:
    print('まだそろっていません（集計と図は出しません）。足りないもの：', missing)
    raise SystemExit('⑪ は、A と B がすべてそろってから実行します')
extra = {'edm_commit': EDM_COMMIT, 'gpu': GPU_NAME, 'torch': torch.__version__}
summ = C6.report6(ROOT3, ROOT6, ref, extra=extra)
print(json.dumps(summ['judgement'], ensure_ascii=False, indent=1, default=str)[:6000])
from IPython.display import Image, display
for f in summ['figures']:
    display(Image(os.path.join(ROOT6, 'figures', f)))
zips, nf = C6.pack6(ROOT6)
print('持ち帰る zip：', zips, '（%d ファイル）' % nf)
